In [ ]:
import os
from pathlib import Path

def _load_local_env(path='.env'):
    env_path = Path(path)
    if not env_path.exists():
        return
    for line in env_path.read_text(errors='replace').splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip().removeprefix('export ').strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value

_load_local_env()
if not os.environ.get('HF_TOKEN') and os.environ.get('Huggingface_token'):
    os.environ['HF_TOKEN'] = os.environ['Huggingface_token']

if not os.environ.get("HF_TOKEN"):
    print("=" * 60)
    print("HF_TOKEN not found in environment.")
    print("1. Get your token: https://huggingface.co/settings/tokens")
    print("2. Accept terms: https://huggingface.co/MahmoodLab/UNI")
    print("3. Set it below (REPLACE with your actual token):")
    print("=" * 60)
    # --- PASTE YOUR TOKEN HERE ---
    # Keep HF_TOKEN unset locally; export a real token before execution.
    # --- ^^^^^^^^^^^^^^^^^^^^^^^^^

print('Dependency install skipped locally; using the project .venv.')

import timm
if timm.__version__ != '0.9.16':
    print(f'WARNING: timm is {timm.__version__}, expected 0.9.16. Restart kernel and re-run.')
else:
    print(f'timm {timm.__version__} confirmed')
print('Dependencies OK.')


In [ ]:
import os, gc, json, warnings, random, copy, math
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('HDF5_USE_FILE_LOCKING', 'FALSE')
os.environ['TORCH_COMPILE_DISABLE'] = '1'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from collections import defaultdict
from itertools import product

import torch
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
torch.set_num_threads(1)
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import timm

from sklearn.metrics import (precision_recall_fscore_support, confusion_matrix,
                             roc_auc_score, cohen_kappa_score, accuracy_score,
                             matthews_corrcoef, average_precision_score)
from sklearn.model_selection import StratifiedKFold
from scipy.special import softmax
from scipy.stats import ttest_rel
import cv2
import PIL.Image

warnings.filterwarnings('ignore')
print('All imports OK.')


In [ ]:
# Configuration -- 3-class production
CFG = {
    'seed': 42,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'num_classes': 3,
    'n_branches': 3,
    'encoder': 'uni',
    'feature_dim': 1024,
    'patch_size': 224,
    'n_folds': 5,
    'n_seeds': 3,   # 5-fold Ã— 3-seed = 15 TransMIL models + 15 baselines (~5h)
    'batch_size': 24,
    'num_workers': 0,
    'mil_lr': 3e-5,
    'mil_weight_decay': 1e-4,
    'mil_patience': 15,
    'early_stop_delta': 0.0005,
    'min_epochs': 20,
    'mil_epochs': 80,
    'dropout': 0.3,
    'max_grad_norm': 1.0,
    'data_dir': 'data/bracs_roi',
    'output_dir': 'work/transmil_dtfd_output',
    'feature_cache': 'work/bracs_features.h5',
    'class_names': ['Benign', 'Atypia', 'Malignant'],
    'class_full': ['Benign (N+PB+UDH)', 'Atypia (FEA+ADH)', 'Malignant (DCIS+IC)'],
    'atypia_threshold': 0.44,
    'review_margin_threshold': 0.10,
    'atypia_boundary_margin': 0.05,
    'class_weight_power': 0.75,
    'class_weight_scales': [1.0, 0.90, 1.0],
    'bootstrap_samples': 1000,
    'mixup_alpha': 0.3,
    'label_smoothing': 0.1,
    'inst_loss_weight': 0.4,
    'feat_norm': True,
    'feat_aug_std': 0.01,
    'feat_dropout': 0.1,
    'attn_drop_post': 0.1,
    'grad_accum': 1,
    'warmup_epochs': 8,
    'n_tta': 4,
    'clam_hidden': 1024,
    'lookahead_k': 5,
    'lookahead_alpha': 0.5,
    'ema_decay': 0.995,
    'max_patches_per_bag': 512,  # patch cap for tissue coverage
    'progressive_start': 512,
    'progressive_epochs': 8,
    'swa_start': 20,
    'swa_lr': 1e-5,
    # TransMIL architecture
    'mil_arch': 'transmil',
    'transmil_dim': 768,
    'transmil_layers': 3,
    'transmil_heads': 8,
    'transmil_landmarks': 64,
    'n_pseudo_bags': 8,
    'focal_gamma': 2.5,
    'multi_scale_sizes': [224, 448, 896],  # multi-scale enabled (3x cache, ~2x runtime)
    'stochastic_depth': 0.1,
    'context_fusion_dim': 512,  # hidden dim for Context Fusion block
    'use_cluster_dtfd': True,  # True = K-Means DTFD pseudo-bags; False = random
    'use_lookahead': True,
    'use_gc': True,
    'use_ema': True,
    'use_swa': True,
    'use_uncertainty_head': False,
    'use_temp_scaling': True,
    'use_ood_distillation': True,
    'distill_weight': 0.5,
    'distill_temp': 4.0,
    'distill_alpha': 0.7,
    'distill_epochs': 30,
}
if CFG['device'] == 'cpu':
    raise RuntimeError('GPU NOT DETECTED -- aborting')
if not os.path.exists(CFG['data_dir']):
    import glob
    potential = sorted(glob.glob(os.path.join(CFG['data_dir'], '**', '*_N_*.png'), recursive=True))
    if potential:
        p = os.path.dirname(potential[0])
        while (str(CFG['data_dir']) + os.sep) in p:
            bn = os.path.basename(p)
            if bn.startswith(('0_','1_','2_','3_','4_','5_','6_')) or bn in ('train','val','test'):
                p = os.path.dirname(p)
            else:
                break
        CFG['data_dir'] = p
        print(f'Data dir auto-detected: {p}')
    else:
        candidates = sorted(glob.glob(os.path.join(CFG['data_dir'], '**/'), recursive=True))
        print(f'No PNGs found: {candidates[:10]}')
        raise RuntimeError('No BRACS PNGs found. Mount novo-bracs-original dataset.')
print(f'Config: {CFG["num_classes"]}-class, {CFG["n_folds"]}-fold, {CFG["n_seeds"]}-seed')
print(f'Device: {CFG["device"]}')




# Dataset

BRACS ROI images are grouped into the three classes used in this project.

| BRACS labels | Project class | Label |
|---|---|---:|
| N, PB, UDH | Benign | 0 |
| FEA, ADH | Atypia | 1 |
| DCIS, IC | Malignant | 2 |

Splits are created at patient level so that the same patient does not appear in both training and evaluation sets.


In [ ]:
# Scan dataset, build metadata (3-class), create patient-disjoint splits
import glob

base_dir = CFG['data_dir']
roi_paths = sorted(glob.glob(os.path.join(base_dir, '**', '*.png'), recursive=True))
if not roi_paths:
    raise RuntimeError('No PNGs found. Place BRACS ROI images under data/bracs_roi/.')
print(f'Total ROIs found: {len(roi_paths)}')

# Deduplicate at ROI level before patient grouping
from collections import Counter
roi_deduped = sorted(set(roi_paths))
print(f'Unique ROIs after dedup: {len(roi_deduped)}')

# 3-class label mapping
class_to_idx = {'N': 0, 'PB': 0, 'UDH': 0, 'FEA': 1, 'ADH': 1, 'DCIS': 2, 'IC': 2}

metadata_records = []
for path in roi_deduped:
    fname = os.path.basename(path)
    parts = fname.replace('.png', '').split('_')
    if len(parts) >= 3 and parts[0] == 'BRACS':
        patient_id = parts[1]
        slide_id = f'{parts[0]}_{parts[1]}_{parts[2]}'
        label = class_to_idx.get(parts[2], 0)
    elif len(parts) >= 3:
        patient_id = parts[0]
        slide_id = f'SLIDE_{parts[0]}_{parts[1]}'
        label = class_to_idx.get(parts[1], 0)
    else:
        continue
    metadata_records.append({'slide_id': slide_id, 'patient_id': patient_id,
                             'label': label, 'label_orig': parts[2] if len(parts) > 2 else parts[1], 'path': path})

df_meta = pd.DataFrame(metadata_records)
df_meta = df_meta.drop_duplicates(subset=['slide_id', 'path'])
n_patients = df_meta['patient_id'].nunique()
print(f'Parsed {len(df_meta)} ROIs across {n_patients} unique patients, {df_meta["slide_id"].nunique()} unique slides')
print(f'Label distribution:\n{df_meta["label"].value_counts().sort_index()}')
print(f'=== DATA LEAKAGE CHECK === Patient-disjoint splits - no patient spans multiple folds or test set ===')

# === HELD-OUT TEST SET (80/20 patient-stratified, untouched during CV) ===
from sklearn.model_selection import train_test_split
patients = df_meta.groupby('patient_id')['label'].max().reset_index()
trainval_patients, test_patients = train_test_split(
    patients, test_size=0.2, random_state=CFG['seed'], stratify=patients['label']
)
trainval_pids = set(trainval_patients['patient_id'])
test_pids = set(test_patients['patient_id'])
trainval_df = df_meta[df_meta['patient_id'].isin(trainval_pids)].copy()
test_df = df_meta[df_meta['patient_id'].isin(test_pids)].copy()
slide_overlap = set(trainval_df['slide_id']) & set(test_df['slide_id'])
patient_overlap = trainval_pids & test_pids
print(f'  Held-out test: {len(test_df)} ROIs ({test_df["slide_id"].nunique()} slides, {len(test_pids)} patients) -- NEVER seen during CV/ablation/distillation')
print(f'  CV pool:       {len(trainval_df)} ROIs ({trainval_df["slide_id"].nunique()} slides, {len(trainval_pids)} patients)')
print(f'  Slide overlap:  {"LEAK!" if slide_overlap else "None - patient-disjoint, safe"}')
print(f'  Patient overlap: {"LEAK!" if patient_overlap else "None"}')
if slide_overlap or patient_overlap:
    raise RuntimeError('Patient/slide leakage detected between CV and held-out test split')
test_df.to_csv(os.path.join('work', 'test_metadata.csv'), index=False)
import json
with open(os.path.join('work', 'test_info.json'), 'w') as _f:
    json.dump({'slides': sorted(test_df['slide_id'].unique().tolist()), 'patients': sorted(test_pids)}, _f)
print(f'  Test label distribution: {dict(test_df["label"].value_counts().sort_index())}')

metadata_csv = os.path.join('work', 'metadata.csv')
trainval_df.to_csv(metadata_csv, index=False)

def create_bracs_splits(metadata_csv, n_folds=5, seed=42):
    df = pd.read_csv(metadata_csv)
    patients = df.groupby('patient_id')['label'].max().reset_index()
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    splits = {}
    for fold, (train_idx, val_idx) in enumerate(skf.split(patients, patients['label'])):
        train_patients = set(patients.iloc[train_idx]['patient_id'])
        val_patients = set(patients.iloc[val_idx]['patient_id'])
        train_slides = sorted(set(df[df['patient_id'].isin(train_patients)]['slide_id'].tolist()))
        val_slides = sorted(set(df[df['patient_id'].isin(val_patients)]['slide_id'].tolist()))
        splits[fold] = {'train': train_slides, 'val': val_slides}
        train_set = set(train_slides)
        val_set = set(val_slides)
        overlap = train_set & val_set
        patient_overlap = train_patients & val_patients
        if overlap or patient_overlap:
            raise RuntimeError(f'Leakage detected in fold {fold}: slides={len(overlap)} patients={len(patient_overlap)}')
        print(f'Fold {fold}: {len(train_set)} train, {len(val_set)} val clean')
    return splits

splits = create_bracs_splits(metadata_csv, n_folds=CFG['n_folds'], seed=CFG['seed'])
for f in range(CFG['n_folds']):
    train_df = trainval_df[trainval_df['slide_id'].isin(splits[f]['train'])]
    val_df = trainval_df[trainval_df['slide_id'].isin(splits[f]['val'])]
    print(f'  Fold {f}: Train {dict(train_df["label"].value_counts().sort_index())}, Val {dict(val_df["label"].value_counts().sort_index())}')
print('Patient-disjoint 3-class splits ready.')


# Feature Extraction

UNI ViT-L is used as a frozen pathology encoder. ROI features are extracted once and cached to HDF5 for MIL training.


In [ ]:
# Patch extraction from ROI
# Features extracted at a unified resolution equivalent to 20x magnification (target_mpp=0.5)
def extract_patches_from_roi(roi_path, target_size=None, target_mpp=0.5):
    if target_size is None:
        target_size = CFG.get('multi_scale_sizes', [224])
    if isinstance(target_size, int):
        target_size = [target_size]
    if isinstance(roi_path, str):
        img = cv2.imread(roi_path)
        if img is None:
            return {s: [] for s in target_size}
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        img = roi_path
    h, w = img.shape[:2]
    if w > 2000 or h > 2000:
        scale = target_mpp / 0.25
        new_w, new_h = max(1, int(w // scale)), max(1, int(h // scale))
        img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
        h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    tissue_mask = (gray > 20).astype(np.float32)
    has_tissue = tissue_mask.mean() > 0.1
    result = {}
    for sz in target_size:
        patches = []
        if h >= sz and w >= sz:
            for y in range(0, h - sz + 1, sz):
                for x in range(0, w - sz + 1, sz):
                    if tissue_mask[y:y+sz, x:x+sz].mean() > 0.1:
                        patch = cv2.resize(img[y:y+sz, x:x+sz], (224, 224), interpolation=cv2.INTER_AREA)
                        patches.append((patch, x, y, sz, sz))
        if not patches and has_tissue:
            patch = cv2.resize(img, (224, 224), interpolation=cv2.INTER_AREA)
            patches.append((patch, 0, 0, w, h))
        result[sz] = patches
    return result
print('Patch extraction ready.')



In [ ]:
# Load UNI encoder

def get_uni_transform():
    from timm.data import resolve_data_config
    from timm.data.transforms_factory import create_transform
    config = resolve_data_config({}, model='vit_large_patch16_224')
    return create_transform(**config)


def _create_uni_model():
    return timm.create_model(
        'vit_large_patch16_224',
        img_size=224, patch_size=16, init_values=1e-5, num_classes=0, dynamic_img_size=True
    )


def _load_uni_state(model, ckpt_path, device):
    state_dict = torch.load(ckpt_path, map_location='cpu')
    if any(k.startswith('module.') for k in state_dict):
        state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict, strict=True)
    model.to(device)
    model.eval()
    return model


def load_uni(device='cuda'):
    local_paths = [os.path.join('weights', 'pytorch_model.bin'), os.path.join('weights', 'uni.pth'), 'uni.pth', 'pytorch_model.bin']
    for path in local_paths:
        if os.path.exists(path):
            model = _create_uni_model()
            model = _load_uni_state(model, path, device)
            print(f'UNI loaded from {path} (1024-d)')
            return model

    try:
        from huggingface_hub import hf_hub_download, login
        _tok = os.environ.get('HF_TOKEN', '')
        if _tok:
            login(token=_tok, add_to_git_credential=False)
        model = _create_uni_model()
        ckpt_path = hf_hub_download('MahmoodLab/UNI', 'pytorch_model.bin')
        model = _load_uni_state(model, ckpt_path, device)
        print('UNI loaded from HF Hub (1024-d)')
        return model
    except Exception as e:
        print(f'HF Hub failed: {e}')
    raise RuntimeError('CRITICAL: UNI weights not found. Download from HF MahmoodLab/UNI.')

print('UNI loader ready.')


In [ ]:
# Extract features to HDF5 cache
def extract_and_cache_features(model, transform, roi_paths, cache_path,
                               slide_label_fn, device=None, batch_size=32):
    device = device or CFG['device']
    import h5py
    batch_size = min(batch_size, 16)
    tmp_path = cache_path + '.tmp'
    if os.path.exists(tmp_path):
        os.remove(tmp_path)
    success = 0
    skip = 0
    with h5py.File(tmp_path, 'a') as h5:
        for path in tqdm(roi_paths, desc='Extracting features'):
            slide_id, label = slide_label_fn(path)
            try:
                patch_dict = extract_patches_from_roi(path, target_size=CFG.get('multi_scale_sizes', [224]))
                scale_keys = sorted(patch_dict.keys())
                has_any = any(len(patch_dict[k]) > 0 for k in scale_keys)
                if not has_any:
                    skip += 1
                    continue
                all_scale_feats = {}
                for sz in scale_keys:
                    patches = patch_dict[sz]
                    if not patches:
                        all_scale_feats[sz] = np.zeros((1, 1024))
                        continue
                    all_feats = []
                    for i in range(0, len(patches), batch_size):
                        batch_patches = patches[i:i+batch_size]
                        imgs = torch.stack([
                            transform(PIL.Image.fromarray(p).convert('RGB'))
                            for p, _, _, _, _ in batch_patches
                        ])
                        with torch.inference_mode(), torch.amp.autocast(device, dtype=torch.float16):
                            feats = model(imgs.to(device)).float().cpu().numpy().astype(np.float32)
                        all_feats.append(feats)
                        del imgs, feats
                        if device == 'cuda':
                            torch.cuda.empty_cache()
                    if all_feats:
                        all_scale_feats[sz] = np.concatenate(all_feats, axis=0)
                    else:
                        all_scale_feats[sz] = np.zeros((1, 1024))
                if slide_id in h5:
                    for sz, feats in all_scale_feats.items():
                        ds_name = f'features_{sz}' if len(scale_keys) > 1 else 'features'
                        if ds_name in h5[slide_id]:
                            existing = h5[slide_id][ds_name][:]
                            combined = np.concatenate([existing, feats], axis=0)
                            h5[slide_id][ds_name].resize((combined.shape[0], feats.shape[1]))
                            h5[slide_id][ds_name][:] = combined
                        else:
                            h5[slide_id].create_dataset(ds_name, data=feats, compression='gzip', maxshape=(None, feats.shape[1]))
                else:
                    grp = h5.create_group(slide_id)
                    for sz, feats in all_scale_feats.items():
                        ds_name = f'features_{sz}' if len(scale_keys) > 1 else 'features'
                        grp.create_dataset(ds_name, data=feats, compression='gzip', maxshape=(None, feats.shape[1]))
                    grp.create_dataset('labels', data=np.array([label]), dtype='int32')
                success += 1
            except Exception as e:
                print(f'Error on {path}: {e}')
                skip += 1
            if device == 'cuda':
                torch.cuda.empty_cache()
    os.replace(tmp_path, cache_path)
    with h5py.File(cache_path, 'r') as _final_h5:
        n_groups = len(_final_h5.keys())
    print(f'Cached features from {success} ROIs into {n_groups} slide groups ({skip} skipped) at {cache_path}')
    return success
print('Feature extraction pipeline ready.')


# Model

The classifier combines TransMIL-style attention with DTFD-MIL feature distillation. Training includes regularization and calibration steps used in the final validation run.

Main components:
- Nystrom self-attention for efficient MIL aggregation
- pseudo-bag feature distillation
- focal loss for class imbalance
- bag-level MixUp and dropout regularization
- EMA/SWA-style checkpoint averaging where enabled


In [ ]:
# TransMIL + DTFD-MIL + Nystrom Attention

class NystromAttention(nn.Module):
    '''Efficient Nystrom-approximated self-attention (O(n) complexity).'''
    def __init__(self, dim, num_heads=8, num_landmarks=64, dropout=0.0):
        super().__init__()
        self.num_heads = num_heads
        self.num_landmarks = num_landmarks
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
        self.attn_drop = nn.Dropout(dropout)
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

    def forward(self, x, mask=None):
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, D // self.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]

        if N <= self.num_landmarks:
            attn = (q @ k.transpose(-2, -1)) * self.scale
            if mask is not None:
                attn = attn.masked_fill(mask[:, None, None, :] == 0, float('-inf'))
            attn = F.softmax(attn, dim=-1)
            attn = self.attn_drop(attn)
            out = attn @ v
        else:
            stride = max(1, N // self.num_landmarks)
            q_landmarks = q[:, :, ::stride, :]
            k_landmarks = k[:, :, ::stride, :]
            qk = (q @ k_landmarks.transpose(-2, -1)) * self.scale
            kq = (k_landmarks @ q_landmarks.transpose(-2, -1)) * self.scale
            if mask is not None:
                m_lm = mask[:, ::stride]
                qk = qk.masked_fill(m_lm[:, None, None, :] == 0, float('-inf'))
                kq = kq.masked_fill(m_lm[:, None, None, :] == 0, float('-inf'))
            qk = F.softmax(qk, dim=-1)
            kq = F.softmax(kq, dim=-1)
            kq_fp32 = kq.float()
            if not torch.isfinite(kq_fp32).all():
                kq_fp32 = torch.nan_to_num(kq_fp32, nan=0.0, posinf=1.0, neginf=0.0)
            try:
                qk_pinv = torch.linalg.pinv(kq_fp32)
            except Exception:
                try:
                    eps = 1e-4
                    identity = torch.eye(kq_fp32.size(-1), device=kq_fp32.device, dtype=kq_fp32.dtype)
                    qk_pinv = torch.linalg.pinv(kq_fp32 + eps * identity)
                except Exception:
                    eps = 1e-3
                    identity = torch.eye(kq_fp32.size(-1), device='cpu', dtype=torch.float32)
                    qk_pinv = torch.linalg.pinv(kq_fp32.cpu() + eps * identity).to(kq_fp32.device)
            qk_pinv = qk_pinv.to(kq.dtype)
            v_tilde = F.softmax((q_landmarks @ k.transpose(-2, -1)) * self.scale, dim=-1) @ v
            out = qk @ qk_pinv @ v_tilde

        out = out.transpose(1, 2).contiguous().view(B, N, D)
        return self.proj(out)


class TransMILBlock(nn.Module):
    def __init__(self, dim=512, num_heads=8, num_landmarks=64, mlp_ratio=4.0, dropout=0.25, sd_prob=0.0):
        super().__init__()
        self.sd_prob = sd_prob
        self.norm1 = nn.LayerNorm(dim)
        self.attn = NystromAttention(dim, num_heads, num_landmarks, dropout)
        self.drop1 = nn.Dropout(dropout)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(int(dim * mlp_ratio), dim),
            nn.Dropout(dropout),
        )

    def forward(self, x, mask=None):
        if self.training and self.sd_prob > 0 and torch.rand(1).item() < self.sd_prob:
            return x
        x = x + self.drop1(self.attn(self.norm1(x), mask))
        x = x + self.mlp(self.norm2(x))
        return x


class TransMIL(nn.Module):
    '''TransMIL: Transformer-based MIL with Nystrom attention + 1D positional encoding.'''
    def __init__(self, in_dim=1024, n_classes=3, dim=512, n_layers=2,
                 num_heads=8, num_landmarks=64, dropout=0.25, max_len=512):
        super().__init__()
        self.dim = dim
        self.fc = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, dim),
            nn.LayerNorm(dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.pos_embed = nn.Parameter(torch.zeros(1, max_len, dim))
        sd = CFG.get('stochastic_depth', 0.0)
        self.blocks = nn.ModuleList([
            TransMILBlock(dim, num_heads, num_landmarks, dropout=dropout,
                          sd_prob=sd * (i / max(n_layers - 1, 1)))
            for i in range(n_layers)
        ])
        self.norm = nn.LayerNorm(dim)
        self.attention = GatedAttention(dim)
        cf_dim = CFG.get('context_fusion_dim', dim)
        self.context_fusion = nn.Sequential(
            nn.Linear(dim, cf_dim),
            nn.LayerNorm(cf_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(cf_dim, dim),
        )
        self.classifier = nn.Linear(dim, n_classes)
        self.uncertainty_head = None

    def forward(self, x, mask=None, return_attn=False):
        if isinstance(x, dict):
            feats_list = []
            sorted_keys = sorted(x.keys())
            for sz in sorted_keys:
                x_sz = x[sz]
                if isinstance(x_sz, (list, tuple)):
                    x_sz = x_sz[0]
                x_sz = self.fc(x_sz) if x_sz.size(-1) != self.dim else x_sz
                feats_list.append(x_sz)
            cat_dim = 1 if feats_list[0].dim() == 3 else 0
            x = torch.cat(feats_list, dim=cat_dim)
            if mask is not None and isinstance(mask, dict):
                mask_list = [mask[sz] for sz in sorted_keys]
                cat_dim_mask = 1 if mask_list[0].dim() == 2 else 0
                mask = torch.cat(mask_list, dim=cat_dim_mask)
            B, N, D = x.shape if x.dim() == 3 else (1, *x.shape)
            if x.dim() == 2:
                x = x.unsqueeze(0)
                if mask is not None: mask = mask.unsqueeze(0)
        else:
            B, N, D = x.shape
            x = self.fc(x)
        if N <= self.pos_embed.size(1):
            x = x + self.pos_embed[:, :N]
        else:
            pos = F.interpolate(self.pos_embed.transpose(1, 2), size=N, mode='linear', align_corners=False).transpose(1, 2)
            x = x + pos
        for block in self.blocks:
            x = block(x, mask)
        x = self.norm(x)
        A = self.attention(x, mask)
        pooled = (x * A).sum(dim=1)
        pooled = self.context_fusion(pooled)
        logits = self.classifier(pooled)
        if self.uncertainty_head is not None and return_attn:
            return logits, A, self.uncertainty_head(pooled)
        if self.uncertainty_head is not None:
            return logits, self.uncertainty_head(pooled)
        if return_attn:
            return logits, A
        return logits





class UncertaintyHead(nn.Module):
    """Trainable uncertainty head: maps bag embedding to per-class uncertainty."""
    def __init__(self, dim, n_classes, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_classes),
        )
    def forward(self, x):
        return F.softplus(self.net(x))

class TemperatureScaling:
    """Post-hoc temperature scaling with a strictly positive temperature."""
    def __init__(self, init_temp=1.0, min_temp=0.05, max_temp=10.0):
        init = torch.tensor([max(float(init_temp), min_temp)], dtype=torch.float32)
        self.raw_temp = nn.Parameter(torch.log(torch.expm1(init).clamp_min(1e-6)))
        self.min_temp = float(min_temp)
        self.max_temp = float(max_temp)

    @property
    def temp(self):
        return F.softplus(self.raw_temp).clamp(min=self.min_temp, max=self.max_temp)

    def fit(self, logits, labels, lr=0.01, max_iter=200):
        logits = logits.detach()
        labels = labels.detach().to(logits.device)
        self.raw_temp.data = self.raw_temp.data.to(logits.device)
        opt = torch.optim.LBFGS([self.raw_temp], lr=lr, max_iter=max_iter)
        def closure():
            opt.zero_grad()
            loss = F.cross_entropy(logits / self.temp, labels)
            loss.backward()
            return loss
        opt.step(closure)

    def __call__(self, logits):
        return logits / self.temp.to(logits.device)

class GatedAttention(nn.Module):
    '''Gated Attention Pooling (Ilse et al., 2018) for TransMIL.'''
    def __init__(self, dim):
        super().__init__()
        self.V = nn.Linear(dim, dim)
        self.U = nn.Linear(dim, dim)
        self.w = nn.Linear(dim, 1)
        
    def forward(self, x, mask=None):
        A = self.w(torch.tanh(self.V(x)) + torch.sigmoid(self.U(x)))
        if mask is not None:
            A = A.masked_fill(mask.unsqueeze(-1) == 0, float('-inf'))
        A = F.softmax(A, dim=1)
        return A


class DTFDWrapper(nn.Module):
    '''DTFD-MIL: pseudo-bag augmentation wrapper around TransMIL.'''
    def __init__(self, base_model, n_pseudo=4):
        super().__init__()
        self.base = base_model
        self.n_pseudo = n_pseudo

    def forward(self, x, mask=None):
        return self.base(x, mask)

    def forward_pseudo(self, x, mask, n_pseudo=None):
        B, N, D = x.shape
        device = x.device
        k = n_pseudo or self.n_pseudo
        all_pseudo, all_masks = [], []
        for b in range(B):
            n_real = mask[b].sum().item()
            if n_real < 2:
                all_pseudo.append(x[b:b+1])
                all_masks.append(mask[b:b+1])
                continue
            real_feats = x[b, :n_real]
            for _ in range(k):
                n_sub = max(1, int(n_real * 0.5))
                idx = torch.randperm(n_real, device=device)[:n_sub]
                pseudo = torch.zeros(1, N, D, device=device)
                pseudo_mask = torch.zeros(1, N, dtype=torch.long, device=device)
                pseudo[0, :n_sub] = real_feats[idx]
                pseudo_mask[0, :n_sub] = 1
                all_pseudo.append(pseudo)
                all_masks.append(pseudo_mask)
        pseudo_x = torch.cat(all_pseudo, dim=0)
        pseudo_m = torch.cat(all_masks, dim=0)
        return self.base(pseudo_x, pseudo_m)


    def forward_pseudo_cluster(self, x, mask, n_pseudo=None):
        '''Cluster-guided DTFD: K-Means on features for meaningful pseudo-bags.'''
        from sklearn.cluster import KMeans
        B, N, D = x.shape
        device = x.device
        k = n_pseudo or self.n_pseudo
        all_pseudo, all_masks = [], []
        for b in range(B):
            n_real = int(mask[b].sum().item())
            if n_real < k + 1:
                for _ in range(k):
                    all_pseudo.append(x[b:b+1])
                    all_masks.append(mask[b:b+1])
                continue
            real_feats = x[b, :n_real].cpu().numpy()
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                try:
                    km = KMeans(n_clusters=k, random_state=42, n_init=1).fit(real_feats)
                    labels = km.labels_
                except Exception:
                    labels = None
            if labels is None:
                for _ in range(k):
                    idx = torch.randperm(n_real, device=device)[:max(1, n_real // k)]
                    pseudo = torch.zeros(1, N, D, device=device)
                    pseudo_mask = torch.zeros(1, N, dtype=torch.long, device=device)
                    pseudo[0, :len(idx)] = x[b, idx]
                    pseudo_mask[0, :len(idx)] = 1
                    all_pseudo.append(pseudo)
                    all_masks.append(pseudo_mask)
            else:
                for ci in range(k):
                    ci_idx = torch.where(torch.tensor(labels == ci, device=device))[0]
                    if len(ci_idx) == 0:
                        ci_idx = torch.randperm(n_real, device=device)[:1]
                    pseudo = torch.zeros(1, N, D, device=device)
                    pseudo_mask = torch.zeros(1, N, dtype=torch.long, device=device)
                    pseudo[0, :len(ci_idx)] = x[b, ci_idx]
                    pseudo_mask[0, :len(ci_idx)] = 1
                    all_pseudo.append(pseudo)
                    all_masks.append(pseudo_mask)
        pseudo_x = torch.cat(all_pseudo, dim=0)
        pseudo_m = torch.cat(all_masks, dim=0)
        return self.base(pseudo_x, pseudo_m)

    def get_base(self):
        return self.base


# Gated attention head (used by CLAM_SB baseline)
class GatedAttnHead(nn.Module):
    def __init__(self, in_dim=512, hidden_dim=256, dropout=0.2):
        super().__init__()
        self.ln = nn.LayerNorm(in_dim)
        self.gate = nn.Linear(in_dim, hidden_dim)
        self.tanh = nn.Linear(in_dim, hidden_dim)
        self.attn_drop = nn.Dropout(dropout)
        self.B = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(0)
        x = self.ln(x)
        A = torch.sigmoid(self.gate(x)) * torch.tanh(self.tanh(x))
        A = self.attn_drop(A)
        A = self.B(A).squeeze(-1)
        A = F.softmax(A, dim=1)
        M = torch.bmm(A.unsqueeze(1), x).squeeze(1)
        return M, A


# Gradient Centralization (Yong et al. 2020)
def gradient_centralization(optimizer):
    for group in optimizer.param_groups:
        for p in group['params']:
            if p.grad is not None and p.dim() > 1:
                p.grad.data.sub_(p.grad.data.mean(dim=tuple(range(1, p.dim())), keepdim=True))


# Stochastic Weight Averaging (SWA)
class SWAWrapper:
    def __init__(self, model, start_epoch=10, swa_lr=2e-5):
        self.model = model
        self.start_epoch = start_epoch
        self.swa_lr = swa_lr
        self.swa_state = {}
        self.n_averaged = 0

    def update(self, epoch, optimizer):
        if epoch + 1 >= self.start_epoch:
            if self.n_averaged == 0:
                self.swa_state = {k: v.detach().clone() for k, v in self.model.state_dict().items()}
                self.n_averaged = 1
                for g in optimizer.param_groups:
                    g['lr'] = self.swa_lr
            else:
                n = self.n_averaged + 1
                for k, v in self.model.state_dict().items():
                    if k in self.swa_state:
                        self.swa_state[k] = (self.swa_state[k] * self.n_averaged + v.detach().clone()) / n
                self.n_averaged = n

    def apply_to(self, target_model):
        if self.n_averaged > 0:
            target_keys = set(target_model.state_dict().keys())
            swa_keys = set(self.swa_state.keys())
            is_target_dp = any(k.startswith('module.') for k in target_keys)
            is_swa_dp = any(k.startswith('module.') for k in swa_keys)
            if is_target_dp and not is_swa_dp:
                params = {'module.' + k: v for k, v in self.swa_state.items()}
            elif not is_target_dp and is_swa_dp:
                params = {k.replace('module.', ''): v for k, v in self.swa_state.items()}
            else:
                params = self.swa_state
            target_model.load_state_dict(params)


class EMAWrapper:
    def __init__(self, model, decay=0.99):
        self.model = model
        self.decay = decay
        self.ema_params = {k: v.detach().clone() for k, v in model.state_dict().items()}

    def update(self):
        with torch.no_grad():
            for k, v in self.model.state_dict().items():
                if k in self.ema_params:
                    self.ema_params[k] = self.decay * self.ema_params[k] + (1 - self.decay) * v
                else:
                    self.ema_params[k] = v.detach().clone()

    def apply_to(self, target_model):
        target_keys = set(target_model.state_dict().keys())
        ema_keys = set(self.ema_params.keys())
        is_target_dp = any(k.startswith('module.') for k in target_keys)
        is_ema_dp = any(k.startswith('module.') for k in ema_keys)
        if is_target_dp and not is_ema_dp:
            params = {'module.' + k: v for k, v in self.ema_params.items()}
        elif not is_target_dp and is_ema_dp:
            params = {k.replace('module.', ''): v for k, v in self.ema_params.items()}
        else:
            params = self.ema_params
        target_model.load_state_dict(params)

    def state_dict(self):
        return self.ema_params

    def load_state_dict(self, state_dict):
        self.ema_params = state_dict


class LookaheadOptimizer:
    def __init__(self, optimizer, k=5, alpha=0.5):
        self.optimizer = optimizer
        self.k = k
        self.alpha = alpha
        self.step_count = 0
        self.param_groups_backup = [
            {'params': [p.clone().detach() for p in group['params']]}
            for group in optimizer.param_groups
        ]

    def zero_grad(self):
        self.optimizer.zero_grad()

    def step(self, closure=None):
        loss = self.optimizer.step(closure)
        self.step_count += 1
        if self.step_count % self.k == 0:
            for group, backup in zip(self.optimizer.param_groups, self.param_groups_backup):
                for p, q in zip(group['params'], backup['params']):
                    p.data.copy_(p.data + self.alpha * (q.data - p.data))
                    q.data.copy_(p.data)
        return loss

    def state_dict(self):
        return {'optimizer': self.optimizer.state_dict(),
                'step_count': self.step_count,
                'backup': self.param_groups_backup}

    def load_state_dict(self, state_dict):
        self.optimizer.load_state_dict(state_dict['optimizer'])
        self.step_count = state_dict['step_count']
        self.param_groups_backup = state_dict['backup']

    @property
    def param_groups(self):
        return self.optimizer.param_groups

    def __getattr__(self, name):
        return getattr(self.optimizer, name)

print('TransMIL + DTFD-MIL + GC + SWA + Lookahead + EMA ready.')



In [ ]:
# MIL Dataset and DataLoader

import hashlib as _hashlib

def _stable_int_seed(*parts):
    payload = '|'.join(map(str, parts)).encode('utf-8')
    return int.from_bytes(_hashlib.blake2b(payload, digest_size=8).digest(), 'big') % (2**31 - 1)

def _stable_patch_subset(feats, max_patches, slide_id, scale):
    if max_patches is None or feats.size(0) <= max_patches:
        return feats
    gen = torch.Generator()
    gen.manual_seed(_stable_int_seed(slide_id, scale, max_patches, CFG.get('seed', 0)))
    idx = torch.randperm(feats.size(0), generator=gen)[:max_patches]
    return feats[idx]

class MILDataset(Dataset):
    def __init__(self, h5_path, slide_ids, class_weights=None, max_patches=None, scale_sizes=None):
        import h5py
        self.slide_ids = []
        self.labels = []
        self.features = {}
        self.max_patches = max_patches
        self.scale_sizes = scale_sizes or CFG.get('multi_scale_sizes', [224])
        self.multi_scale = len(self.scale_sizes) > 1
        with h5py.File(h5_path, 'r') as f:
            for sid in slide_ids:
                if sid not in f:
                    continue
                if self.multi_scale:
                    feats_dict = {}
                    for sz in self.scale_sizes:
                        ds_name = f'features_{sz}'
                        if ds_name in f[sid]:
                            feats = torch.FloatTensor(f[sid][ds_name][:])
                        else:
                            feats = torch.FloatTensor(f[sid]['features'][:])
                        feats = _stable_patch_subset(feats, max_patches, sid, sz)
                        feats_dict[str(sz)] = feats
                    self.features[sid] = feats_dict
                else:
                    ds_name = f'features_{self.scale_sizes[0]}'
                    if ds_name in f[sid]:
                        feats = torch.FloatTensor(f[sid][ds_name][:])
                    else:
                        feats = torch.FloatTensor(f[sid]['features'][:])
                    feats = _stable_patch_subset(feats, max_patches, sid, self.scale_sizes[0])
                    self.features[sid] = feats
                self.labels.append(int(f[sid]['labels'][0]))
                self.slide_ids.append(sid)
        self.labels = np.array(self.labels)
        if class_weights is None:
            counts = np.bincount(self.labels, minlength=CFG['num_classes'])
            n = len(self.labels)
            counts_safe = np.maximum(counts.astype(float), 1.0)
            weights = n / (CFG['num_classes'] * counts_safe)
            weights = np.power(weights, float(CFG.get('class_weight_power', 1.0)))
            scales = np.asarray(CFG.get('class_weight_scales', [1.0] * CFG['num_classes']), dtype=float)
            if len(scales) == CFG['num_classes']:
                weights = weights * scales
            weights = weights * (CFG['num_classes'] / max(weights.sum(), 1e-8))
            self.class_weights = torch.FloatTensor(weights)
        else:
            self.class_weights = class_weights
        _n_patches = 0
        for _f in self.features.values():
            if isinstance(_f, dict):
                _n_patches += sum(t.size(0) for t in _f.values())
            else:
                _n_patches += _f.size(0)
        print(f'  Loaded {len(self.slide_ids)} bags ({_n_patches} patches)')

    def __len__(self):
        return len(self.slide_ids)

    def __getitem__(self, idx):
        sid = self.slide_ids[idx]
        feats = self.features[sid]
        if CFG.get('feat_norm', True):
            if isinstance(feats, dict):
                feats = {k: F.normalize(v, p=2, dim=1) for k, v in feats.items()}
            else:
                feats = F.normalize(feats, p=2, dim=1)
        return feats, self.labels[idx], sid

    def get_class_weights(self):
        return self.class_weights


def mil_collate(batch):
    feats_list, labels_list, ids_list = zip(*batch)
    return list(feats_list), torch.LongTensor(labels_list), list(ids_list)

def padded_collate_multiscale(batch, max_patches=512):
    '''Collate for multi-scale: returns dict of padded tensors per scale.'''
    feats_list, labels_list, ids_list = zip(*batch)
    if isinstance(feats_list[0], dict):
        scales = feats_list[0].keys()
        result = {}
        for k in scales:
            scale_batch = [(f[k], lbl, sid) for f, lbl, sid in zip(feats_list, labels_list, ids_list)]
            result[k] = padded_collate(scale_batch, max_patches)
        feat_tensors = {k: v[0] for k, v in result.items()}
        label_tensor = result[list(scales)[0]][1]
        masks = {k: v[3] for k, v in result.items() if len(v) > 3}
        ids = result[list(scales)[0]][2] if len(result[list(scales)[0]]) > 2 else list(ids_list)
        return feat_tensors, label_tensor, ids, masks if isinstance(masks, dict) else None
    return padded_collate(batch, max_patches)


def padded_collate(batch, max_patches=512):
    feats_list, labels_list, ids_list = zip(*batch)
    batch_size = len(feats_list)
    feat_dim = feats_list[0].size(1) if feats_list[0].dim() > 1 else 1
    actual_lens = [min(f.size(0), max_patches) for f in feats_list]
    max_len = min(max(max(actual_lens), 1), max_patches)
    padded = torch.zeros(batch_size, max_len, feat_dim)
    mask = torch.zeros(batch_size, max_len, dtype=torch.long)
    for i, (feats, n) in enumerate(zip(feats_list, actual_lens)):
        if feats.size(0) > max_patches:
            feats = _stable_patch_subset(feats, max_patches, ids_list[i], 'collate')
        padded[i, :n] = feats[:n]
        mask[i, :n] = 1
    return padded, torch.LongTensor(labels_list), list(ids_list), mask

print('MIL Dataset + collate ready.')


In [ ]:
# Training loop
def compute_loss(logits, labels, class_weights=None, lsmooth=0.0, focal_gamma=0.0):
    '''Cross-entropy with optional focal weighting + label smoothing.'''
    if focal_gamma > 0:
        ce = F.cross_entropy(logits, labels, weight=class_weights, reduction='none')
        pt = torch.exp(-ce)
        loss = ((1 - pt) ** focal_gamma * ce).mean()
    else:
        loss = F.cross_entropy(logits, labels, weight=class_weights)
    if lsmooth > 0:
        n_cls = logits.size(1)
        smooth = torch.full_like(logits, lsmooth / (n_cls - 1))
        smooth[range(len(labels)), labels] = 1.0 - lsmooth
        kl = F.kl_div(F.log_softmax(logits, dim=-1), smooth, reduction='batchmean')
        loss = 0.9 * loss + 0.1 * kl
    return loss

def uncertainty_regularized_loss(logits, labels, class_weights, lsmooth, focal_gamma, uncertainty=None):
    if uncertainty is not None:
        u_loss = uncertainty[range(len(labels)), labels].mean() * 0.1
    else:
        u_loss = 0.0
    return compute_loss(logits, labels, class_weights, lsmooth, focal_gamma) + u_loss

def mixup_bags(feats, mask, labels, alpha=0.2):
    '''Bag-level MixUp augmentation for padded MIL batches.'''
    if alpha <= 0:
        return feats, mask.float(), labels, labels, 1.0
    B = feats.size(0)
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(B, device=feats.device)
    mixed = lam * feats + (1 - lam) * feats[idx]
    mixed_mask = (mask.float() + mask[idx].float()).clamp(0, 1)
    return mixed, mixed_mask, labels, labels[idx], lam
def train_epoch(model, loader, optimizer, scheduler=None, device=None,
                class_weights=None, epoch=0, ema=None, scaler=None, swa=None):
    device = device or CFG['device']
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []
    lsmooth = CFG.get('label_smoothing', 0.0)
    focal_gamma = CFG.get('focal_gamma', 0.0)
    mixup_alpha = CFG.get('mixup_alpha', 0.0)
    warmup_epochs = CFG.get('warmup_epochs', 0)
    n_pseudo = CFG.get('n_pseudo_bags', 0)
    has_dtfd = n_pseudo > 0 and isinstance(model, (DTFDWrapper, nn.DataParallel))
    base_dropout = CFG['dropout']
    dropout_scale = min(1.0, (epoch + 1) / 20)
    for module in model.modules():
        if isinstance(module, nn.Dropout):
            module.p = base_dropout * dropout_scale
    for (feats_padded, labels_list, _, mask) in tqdm(loader, desc='Train', leave=False, mininterval=30.0):
        if isinstance(feats_padded, dict):
            feats_padded = {k: v.to(device, non_blocking=True) for k, v in feats_padded.items()}
            mask = {k: v.to(device, non_blocking=True) for k, v in mask.items()}
        else:
            feats_padded = feats_padded.to(device, non_blocking=True)
            mask = mask.to(device, non_blocking=True)
        labels_list = labels_list.to(device)
        if CFG['feat_aug_std'] > 0:
            if isinstance(feats_padded, dict):
                for k in feats_padded:
                    noise = torch.randn_like(feats_padded[k]) * CFG['feat_aug_std']
                    if isinstance(mask, dict):
                        feats_padded[k] = feats_padded[k] + noise * mask[k].unsqueeze(-1).float()
                    else:
                        feats_padded[k] = feats_padded[k] + noise * mask.unsqueeze(-1).float()
            else:
                noise = torch.randn_like(feats_padded) * CFG['feat_aug_std']
                feats_padded = feats_padded + noise * mask.unsqueeze(-1).float()
        optimizer.zero_grad()
        is_ms = isinstance(feats_padded, dict)
        fwd_feats = feats_padded[min(feats_padded.keys())] if is_ms else feats_padded
        fwd_mask = mask[min(mask.keys())] if is_ms else mask
        if has_dtfd and np.random.random() < 0.5 and not is_ms:
            unwrapped = model.module if isinstance(model, nn.DataParallel) else model
            if CFG.get('use_cluster_dtfd', False):
                logits = unwrapped.forward_pseudo_cluster(fwd_feats, fwd_mask, n_pseudo)
            else:
                logits = unwrapped.forward_pseudo(fwd_feats, fwd_mask, n_pseudo)
            labels_expanded = labels_list.repeat_interleave(n_pseudo)
            loss = compute_loss(logits, labels_expanded[:logits.size(0)], class_weights, lsmooth, focal_gamma)
        elif np.random.random() < 0.5 and mixup_alpha > 0 and not is_ms:
            mixed, mixed_mask, la, lb, lam = mixup_bags(fwd_feats, fwd_mask, labels_list, mixup_alpha)
            logits = model(mixed, mixed_mask)
            loss = lam * compute_loss(logits, la, class_weights, lsmooth, focal_gamma) + (1 - lam) * compute_loss(logits, lb, class_weights, lsmooth, focal_gamma)
        else:
            logits = model(feats_padded, mask)
            if isinstance(logits, (list, tuple)):
                logits = logits[0]
            loss = compute_loss(logits, labels_list, class_weights, lsmooth, focal_gamma)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        if CFG['use_gc']: gradient_centralization(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['max_grad_norm'])
        scaler.step(optimizer)
        scaler.update()
        if ema is not None: ema.update()
        # scheduler stepped once per epoch in train_single_fold
        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels_list[:preds.size(0)].cpu().numpy())
    avg_loss = total_loss / max(len(loader), 1)
    acc = (np.array(all_preds) == np.array(all_labels)).mean() if all_labels else 0.0
    return avg_loss, acc
@torch.inference_mode()
def eval_epoch(model, loader, device=None, class_weights=None, use_ema=None, use_swa=None):
    device = device or CFG['device']
    saved_state = None
    try:
        if use_ema is not None:
            saved_state = copy.deepcopy(model.state_dict())
            use_ema.apply_to(model)
        if use_swa is not None and getattr(use_swa, 'n_averaged', 0) > 0:
            if saved_state is None:
                saved_state = copy.deepcopy(model.state_dict())
            use_swa.apply_to(model)
        model.eval()
        all_preds, all_probs, all_labels, all_ids = [], [], [], []
        n_tta = int(CFG.get('n_tta', 1))
        eval_aug_std = float(CFG.get('eval_feat_aug_std', CFG.get('feat_aug_std', 0.0)))
        base_seed = int(CFG.get('seed', 0))
        for feats_padded, labels_list, ids_list, mask in tqdm(loader, desc='Eval', leave=False, mininterval=30.0):
            if isinstance(feats_padded, dict):
                feats_padded = {k: v.to(device, non_blocking=True) for k, v in feats_padded.items()}
                mask = {k: v.to(device, non_blocking=True) for k, v in mask.items()}
            else:
                feats_padded = feats_padded.to(device, non_blocking=True)
                mask = mask.to(device, non_blocking=True)
            tta_logits = []
            for tta_idx in range(max(n_tta, 1)):
                if eval_aug_std > 0:
                    torch.manual_seed(base_seed + 1009 * tta_idx)
                    if torch.cuda.is_available():
                        torch.cuda.manual_seed_all(base_seed + 1009 * tta_idx)
                if isinstance(feats_padded, dict):
                    f = {k: v.clone() for k, v in feats_padded.items()}
                    if eval_aug_std > 0:
                        f = {k: v + torch.randn_like(v) * eval_aug_std * mask[k].unsqueeze(-1).float() for k, v in f.items()}
                else:
                    f = feats_padded.clone()
                    if eval_aug_std > 0:
                        f = f + torch.randn_like(f) * eval_aug_std * mask.unsqueeze(-1).float()
                out = model(f, mask)
                if isinstance(out, (list, tuple)):
                    out = out[0]
                tta_logits.append(F.softmax(out, dim=-1))
            avg_probs = torch.stack(tta_logits).mean(0)
            all_probs.append(avg_probs.cpu().numpy())
            all_preds.append(avg_probs.argmax(dim=1).cpu().numpy())
            all_labels.append(labels_list.numpy())
            all_ids.extend(ids_list)
        return np.concatenate(all_preds), np.concatenate(all_probs), np.concatenate(all_labels), all_ids
    finally:
        if saved_state is not None:
            model.load_state_dict(saved_state)

print('Train/eval loops with padded batches + DTFD + GC ready.')






# Evaluation

The evaluation reports classification, calibration, and clinical-review metrics.

| Metric group | Examples |
|---|---|
| Classification | Accuracy, macro-F1, weighted-F1 |
| Agreement | Cohen's kappa, MCC |
| Calibration | ECE, Brier score, temperature scaling |
| Clinical review | Sensitivity, specificity, confidence margin |
| Curves | ROC-AUC, PR-AUC |


In [ ]:
# === Phase 3: Explainability Layer ===
# Clinical-grade heatmaps, top-10 patches, error analysis dashboard

import matplotlib.patches as patches

# === Metrics, Uncertainty, and Statistical Tests ===
def predictive_entropy(probs):
    eps = 1e-12
    probs_clipped = np.clip(probs, 1e-12, 1.0)
    return -np.sum(probs * np.log(probs_clipped), axis=1)

def compute_all_metrics(all_labels, all_preds, all_probs, class_names, n_bootstrap=0):
    all_labels = np.asarray(all_labels)
    all_preds = np.asarray(all_preds)
    all_probs = np.asarray(all_probs)
    n_classes = len(class_names)
    p, r, f, s = precision_recall_fscore_support(
        all_labels, all_preds, labels=range(n_classes), zero_division=0)
    _base_macro_f1 = precision_recall_fscore_support(
        all_labels, all_preds, average='macro', zero_division=0)[2]
    metrics = {
        'accuracy': round(accuracy_score(all_labels, all_preds), 4),
        'macro_f1': round(_base_macro_f1, 4),
        'weighted_f1': round(precision_recall_fscore_support(
            all_labels, all_preds, average='weighted', zero_division=0)[2], 4),
        'macro_f1_ci_lo': round(_base_macro_f1, 4),
        'macro_f1_ci_hi': round(_base_macro_f1, 4),
    }
    if n_bootstrap and n_bootstrap > 0 and len(all_labels) > 1:
        rng = np.random.default_rng(CFG.get('seed', 42) if 'CFG' in globals() else 42)
        boot_scores = []
        idx_all = np.arange(len(all_labels))
        n_present = len(np.unique(all_labels))
        for _ in range(int(n_bootstrap)):
            idx = rng.choice(idx_all, size=len(idx_all), replace=True)
            if len(np.unique(all_labels[idx])) < n_present:
                continue
            boot_scores.append(precision_recall_fscore_support(
                all_labels[idx], all_preds[idx], average='macro', zero_division=0)[2])
        if boot_scores:
            metrics['macro_f1_ci_lo'] = round(float(np.percentile(boot_scores, 2.5)), 4)
            metrics['macro_f1_ci_hi'] = round(float(np.percentile(boot_scores, 97.5)), 4)
    per_class = {}
    for i, name in enumerate(class_names):
        y_true_bin = (all_labels == i).astype(int)
        y_pred_bin = (all_preds == i).astype(int)
        pi, ri, fi, _ = precision_recall_fscore_support(
            y_true_bin, y_pred_bin, average='binary', zero_division=0, pos_label=1)
        tn = ((all_labels != i) & (all_preds != i)).sum()
        fp = ((all_labels != i) & (all_preds == i)).sum()
        spec = tn / max(tn + fp, 1)
        support = (all_labels == i).sum()
        try:
            auc = roc_auc_score(y_true_bin, all_probs[:, i])
        except Exception:
            auc = 0.5
        per_class[name] = {
            'precision': round(pi, 4), 'recall': round(ri, 4), 'f1': round(fi, 4),
            'auc': round(auc, 4), 'sensitivity': round(ri, 4),
            'specificity': round(spec, 4), 'support': int(support),
        }
    return metrics, per_class

def paired_ttest(y_true, preds_model, preds_baseline):
    acc_model = (np.array(preds_model) == np.array(y_true)).astype(float)
    acc_baseline = (np.array(preds_baseline) == np.array(y_true)).astype(float)
    t_stat, p_val = ttest_rel(acc_model, acc_baseline)
    return t_stat, p_val

def expected_calibration_error(probs, labels, n_bins=10):
    confidences = np.max(probs, axis=1)
    predictions = np.argmax(probs, axis=1)
    accuracies = (predictions == labels).astype(float)
    ece = 0.0
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    for i in range(n_bins):
        in_bin = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i+1])
        if in_bin.any():
            ece += np.abs(accuracies[in_bin].mean() - confidences[in_bin].mean()) * in_bin.sum() / len(labels)
    return ece

def brier_score(probs, labels, n_classes):
    one_hot = np.eye(n_classes)[labels]
    return np.mean(np.sum((probs - one_hot) ** 2, axis=1))

def sensitivity_specificity(cm):
    n = cm.shape[0]
    sens, spec = [], []
    for i in range(n):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fp - fn
        sens.append(tp / max(tp + fn, 1e-8))
        spec.append(tn / max(tn + fp, 1e-8))
    return sens, spec

def ensemble_uncertainty(all_probs_list):
    stacked = np.stack(all_probs_list, axis=0)
    mean_probs = stacked.mean(axis=0)
    entropy = predictive_entropy(mean_probs)
    variance = stacked.var(axis=0).mean(axis=1)
    return mean_probs, entropy, variance

def confidence_bucket(entropy, n_classes=3):
    max_ent = np.log(n_classes)
    ratios = entropy / max_ent
    cats = np.where(ratios < 0.2, 'High',
           np.where(ratios < 0.5, 'Medium', 'Low'))
    return cats

def mcnemar_test(y_true, preds_a, preds_b):
    from scipy.stats import chi2
    n01 = np.sum((preds_a == y_true) & (preds_b != y_true))
    n10 = np.sum((preds_a != y_true) & (preds_b == y_true))
    chi2_stat = (abs(n01 - n10) - 1) ** 2 / max(n01 + n10, 1)
    p_value = 1 - chi2.cdf(chi2_stat, 1)
    return chi2_stat, p_value, n01, n10

def wilcoxon_paired(metric_a, metric_b):
    from scipy.stats import wilcoxon
    diffs = np.array(metric_a) - np.array(metric_b)
    diffs = diffs[diffs != 0]
    if len(diffs) < 2:
        return 0.0, 1.0
    try:
        stat, p = wilcoxon(diffs, alternative='two-sided')
        return stat, p
    except Exception:
        return 0.0, 1.0

def extract_attention(model, loader, device=None):
    device = device or CFG['device']
    model.eval()
    attn_records = {}
    for feats_padded, labels_list, ids_list, mask in tqdm(loader, desc='Attention', leave=False, mininterval=30.0):
        if isinstance(feats_padded, dict):
            feats_padded = {k: v.to(device, non_blocking=True) for k, v in feats_padded.items()}
            mask = {k: v.to(device, non_blocking=True) for k, v in mask.items()}
        else:
            feats_padded = feats_padded.to(device, non_blocking=True)
            mask = mask.to(device, non_blocking=True)
        with torch.inference_mode(), torch.amp.autocast(device, dtype=torch.float16):
            unwrapped = model.module if isinstance(model, nn.DataParallel) else model
            logits, attn = unwrapped(feats_padded, mask, return_attn=True)
        attn = attn.cpu().numpy()
        for i, sid in enumerate(ids_list):
            if isinstance(mask, dict):
                # Use first/any scale mask for shape; multi-scale handling simplified
                mask_any = mask[min(mask.keys())]
            else:
                mask_any = mask
            n_real = int(mask_any[i].sum().item())
            scores = attn[i, :n_real, 0]
            probs = F.softmax(logits[i], dim=-1).cpu().numpy()
            attn_records[sid] = {
                'attention': scores,
                'prediction': int(logits[i].argmax().item()),
                'label': int(labels_list[i].item()) if hasattr(labels_list[i], 'item') else int(labels_list[i]),
                'probability': probs,
                'confidence': float(np.max(probs)),
                'entropy': float(predictive_entropy(probs.reshape(1, -1))[0]),
            }
    return attn_records


def generate_error_report(all_labels, all_preds, all_probs, all_ids, attn_records,
                          class_names, output_dir, meta_df=None):
    err_dir = os.path.join(output_dir, 'error_analysis')
    os.makedirs(err_dir, exist_ok=True)
    errors = []
    path_map = {}
    if meta_df is not None:
        for _, row in meta_df.iterrows():
            path_map[row['slide_id']] = row.get('path', '')
    for sid, gt, pred, prob in zip(all_ids, all_labels, all_preds, all_probs):
        if gt != pred:
            attn_info = attn_records.get(sid, {})
            attn_scores = attn_info.get('attention', None)
            confidence = float(np.max(prob))
            ent = float(predictive_entropy(prob.reshape(1, -1))[0])
            top_indices = []
            if attn_scores is not None and len(attn_scores) > 0:
                top_indices = np.argsort(attn_scores)[-10:][::-1].tolist()
            errors.append({
                'slide_id': sid,
                'ground_truth': class_names[gt],
                'prediction': class_names[pred],
                'confidence': confidence,
                'entropy': ent,
                'top_attn_patches': str(top_indices),
                'roi_path': path_map.get(sid, ''),
            })
    err_df = pd.DataFrame(errors)
    if len(err_df) > 0:
        err_df.to_csv(os.path.join(output_dir, 'error_analysis.csv'), index=False)
    print(f'  Error analysis: {len(err_df)} misclassified slides logged')
    return err_df


def generate_clinical_explainability_report(roi_path, slide_id, attn_records, cfg, save_dir='clinical_reports'):
    if slide_id not in attn_records:
        print(f"  Skipping {slide_id}: No attention records found.")
        return
    os.makedirs(save_dir, exist_ok=True)
    attn_scores = attn_records[slide_id]['attention']
    pred_class = attn_records[slide_id]['prediction']
    probs = attn_records[slide_id]['probability']
    target_size = cfg['patch_size']
    attn_scores = (attn_scores - attn_scores.min()) / (attn_scores.max() - attn_scores.min() + 1e-8)
    img = cv2.imread(roi_path)
    if img is None:
        print(f"  Error loading {roi_path}")
        return
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if w > 2000 or h > 2000:
        scale = 0.5 / 0.25
        new_w, new_h = int(w // scale), int(h // scale)
        img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
        h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    tissue_mask = (gray > 20).astype(np.float32)
    patch_coords = []
    for y in range(0, h - target_size + 1, target_size):
        for x in range(0, w - target_size + 1, target_size):
            if tissue_mask[y:y+target_size, x:x+target_size].mean() > 0.1:
                patch_coords.append((x, y))
    num_patches = min(len(patch_coords), len(attn_scores))
    patch_coords = patch_coords[:num_patches]
    attn_scores = attn_scores[:num_patches]
    heatmap = np.zeros((h, w), dtype=np.float32)
    for (x, y), score in zip(patch_coords, attn_scores):
        heatmap[y:y+target_size, x:x+target_size] = score
    heatmap_uint8 = np.uint8(255 * heatmap)
    colormap = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    colormap = cv2.cvtColor(colormap, cv2.COLOR_BGR2RGB)
    alpha = np.where(heatmap > 0, 0.45, 0.0)[..., np.newaxis]
    overlay = (img * (1 - alpha) + colormap * alpha).astype(np.uint8)
    top_indices = np.argsort(attn_scores)[-10:][::-1]
    fig = plt.figure(figsize=(24, 10))
    gs = fig.add_gridspec(2, 9)
    ax_main = fig.add_subplot(gs[:, 0:4])
    ax_main.imshow(overlay)
    class_name = cfg['class_names'][pred_class]
    ax_main.set_title(f"ROI Heatmap: {slide_id}\nModel Prediction: {class_name} (Confidence: {np.max(probs):.2f})", fontsize=16, weight='bold')
    ax_main.axis('off')
    for idx, patch_idx in enumerate(top_indices):
        x, y = patch_coords[patch_idx]
        patch_img = img[y:y+target_size, x:x+target_size]
        row, col = divmod(idx, 5)
        ax_patch = fig.add_subplot(gs[row, col + 4])
        ax_patch.imshow(patch_img)
        ax_patch.set_title(f"Rank {idx+1}\nAttn: {attn_scores[patch_idx]:.3f}", fontsize=12)
        ax_patch.axis('off')
        rect = patches.Rectangle((x, y), target_size, target_size, linewidth=2.5, edgecolor='#00FF00', facecolor='none')
        ax_main.add_patch(rect)
        ax_main.text(x, y - 10, str(idx+1), color='#00FF00', fontsize=14, weight='bold',
                     bbox=dict(facecolor='black', alpha=0.5, edgecolor='none', pad=1))
    plt.tight_layout()
    save_path = os.path.join(save_dir, f"{slide_id}_explainability.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    return save_path


def attention_ranking_summary(attn_records, output_dir):
    stats = []
    for sid, rec in attn_records.items():
        attn = rec.get('attention', np.array([0]))
        if len(attn) == 0:
            continue
        stats.append({
            'slide_id': sid,
            'prediction': rec.get('prediction', -1),
            'n_patches': len(attn),
            'mean_attention': float(attn.mean()),
            'std_attention': float(attn.std()),
            'max_attention': float(attn.max()),
            'min_attention': float(attn.min()),
            'top1_idx': int(np.argmax(attn)),
            'top1_score': float(attn.max()),
            'confidence': rec.get('confidence', 0),
            'entropy': rec.get('entropy', 0),
        })
    stat_df = pd.DataFrame(stats)
    stat_df.to_csv(os.path.join(output_dir, 'attention_ranking.csv'), index=False)
    print(f'  Attention ranking: {len(stat_df)} slides')
    return stat_df


def weighted_ensemble(model_predictions, weights=None):
    """Weighted probability averaging across models on the common slide set."""
    if not model_predictions:
        raise ValueError('No model predictions provided for ensemble')
    if weights is None:
        weights = {name: max(info.get('val_f1', 0.5), 0.01) for name, info in model_predictions.items()}
    id_sets = [set(info.get('ids', [])) for info in model_predictions.values()]
    common_ids = sorted(set.intersection(*id_sets)) if id_sets else []
    if not common_ids:
        raise ValueError('No common slide IDs across ensemble inputs')
    dropped = {name: len(set(info.get('ids', [])) - set(common_ids)) for name, info in model_predictions.items()}
    if any(dropped.values()):
        print(f'  Ensemble alignment: using {len(common_ids)} common slides; dropped non-common counts={dropped}')
    total_w = sum(weights.values()) + 1e-8
    weights = {k: v / total_w for k, v in weights.items()}
    all_probs, all_labels = [], []
    for sid in common_ids:
        weighted_sum = None
        label = None
        used_w = 0.0
        for name, info in model_predictions.items():
            id_to_idx = info.get('_id_to_idx')
            if id_to_idx is None:
                id_to_idx = {x: i for i, x in enumerate(info['ids'])}
                info['_id_to_idx'] = id_to_idx
            idx = id_to_idx[sid]
            prob = np.asarray(info['probs'][idx], dtype=float)
            w = weights[name]
            weighted_sum = prob * w if weighted_sum is None else weighted_sum + prob * w
            used_w += w
            if 'labels' in info and len(info['labels']) == len(info['ids']) and label is None:
                label = int(info['labels'][idx])
        all_probs.append(weighted_sum / max(used_w, 1e-8))
        all_labels.append(-1 if label is None else label)
    all_probs = np.asarray(all_probs)
    all_preds = all_probs.argmax(axis=1)
    all_labels = np.asarray(all_labels)
    return all_probs, all_preds, common_ids, all_labels

def delong_roc_variance(y_true, y_scores):
    from scipy.stats import norm
    n_pos = int(np.sum(y_true == 1))
    n_neg = int(np.sum(y_true == 0))
    if n_pos == 0 or n_neg == 0:
        return 0.5, 0.0
    y = y_scores.ravel()
    idx = np.argsort(y)
    y = y[idx]
    gt = y_true[idx]
    xy = y[gt == 1]
    nx = y[gt == 0]
    V10 = 0.0
    V01 = 0.0
    for i in range(len(xy)):
        V10 += (np.sum(nx < xy[i]) + 0.5 * np.sum(nx == xy[i])) / n_neg
    V10 = V10 / n_pos
    for i in range(len(nx)):
        V01 += (np.sum(xy > nx[i]) + 0.5 * np.sum(xy == nx[i])) / n_pos
    V01 = V01 / n_neg
    s2 = (V10 * (1 - V10) + V01 * (1 - V01)) / (n_pos * n_neg)
    auc = np.trapz([np.mean(nx < x) for x in xy]) / (n_pos * n_neg) if len(xy) > 0 else 0.5
    return auc, np.sqrt(max(s2, 1e-10))

def delong_test(y_true, scores_a, scores_b):
    auc1, var1 = delong_roc_variance(y_true, scores_a)
    auc2, var2 = delong_roc_variance(y_true, scores_b)
    from scipy.stats import norm
    z = (auc1 - auc2) / np.sqrt(max(var1 + var2, 1e-10))
    p = 2 * norm.sf(abs(z))
    return auc1, auc2, z, p


def generate_clinical_report(all_labels, all_preds, all_probs, all_ids, attn_records, class_names, output_dir, meta_df=None):
    path_map = {}
    if meta_df is not None:
        for _, r in meta_df.iterrows():
            path_map[r['slide_id']] = r.get('path', '')
    rows = []
    for sid, gt, pred, prob in zip(all_ids, all_labels, all_preds, all_probs):
        attn_info = attn_records.get(sid, {})
        attn_scores = attn_info.get('attention', None)
        confidence = float(np.max(prob))
        ent = float(predictive_entropy(prob.reshape(1, -1))[0])
        top_indices, top_scores = [], []
        if attn_scores is not None and len(attn_scores) > 0:
            ti = np.argsort(attn_scores)[-5:][::-1]
            top_indices = ti.tolist()
            top_scores = [float(attn_scores[i]) for i in ti]
        rows.append({
            'slide_id': sid,
            'true_label': class_names[gt],
            'true_class_id': int(gt),
            'predicted_label': class_names[pred],
            'predicted_class_id': int(pred),
            'correct': int(gt == pred),
            'confidence': confidence,
            'uncertainty_entropy': ent,
            'top5_attn_indices': str(top_indices),
            'top5_attn_scores': str([round(s, 4) for s in top_scores]),
            'roi_path': path_map.get(sid, ''),
        })
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(output_dir, 'clinical_report.csv'), index=False)
    print(f'  Clinical report: {len(df)} slides ({df.correct.sum()} correct, {(~df.correct.astype(bool)).sum()} errors)')
    # Explanation summaries for high-impact cases
    explain_rows = []
    for _, r in df.iterrows():
        if r['correct'] and r['uncertainty_entropy'] < 0.3:
            continue
        if r['correct']:
            summary = f"Slide {r['slide_id']}: correctly classified as {r['predicted_label']} (confidence={r['confidence']:.2f}, uncertainty={r['uncertainty_entropy']:.3f}). Top patches show expected morphological patterns for this class."
        elif r['uncertainty_entropy'] > 0.8:
            summary = f"Slide {r['slide_id']}: MISCLASSIFIED as {r['predicted_label']} (GT={r['true_label']}) with HIGH uncertainty ({r['uncertainty_entropy']:.3f}). Top attention patches may show ambiguous or borderline morphology. Recommend pathologist review and IHC confirmation."
        else:
            summary = f"Slide {r['slide_id']}: MISCLASSIFIED as {r['predicted_label']} (GT={r['true_label']}) with confidence={r['confidence']:.2f}. Top attention patches may contain diagnostically challenging regions."
        explain_rows.append({'slide_id': r['slide_id'], 'explanation': summary})
    exp_df = pd.DataFrame(explain_rows)
    exp_df.to_csv(os.path.join(output_dir, 'clinical_explanations.csv'), index=False)
    print(f'  Clinical explanations: {len(exp_df)} high-impact cases summarized')
    return df, exp_df


print('Phase 3 explainability module ready.')





In [ ]:
# Plotting utilities

def plot_confusion_matrix(cm, class_names, title='Confusion Matrix', save_path=None):
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.ylabel('True')
    plt.xlabel('Predicted')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved {save_path}')
    plt.show()


def plot_roc_curves(all_labels, all_probs, class_names, save_path=None):
    from sklearn.metrics import roc_curve
    plt.figure(figsize=(8, 6))
    for i, name in enumerate(class_names):
        fpr, tpr, _ = roc_curve((all_labels == i).astype(int), all_probs[:, i])
        auc = roc_auc_score((all_labels == i).astype(int), all_probs[:, i])
        plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
    plt.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves')
    plt.legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved {save_path}')
    plt.show()


def plot_calibration_curve(all_probs, all_labels, n_bins=10, save_path=None):
    confidences = np.max(all_probs, axis=1)
    predictions = np.argmax(all_probs, axis=1)
    accuracies = (predictions == all_labels).astype(float)
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    bin_accs = []
    bin_confs = []
    for i in range(n_bins):
        in_bin = (confidences > bin_boundaries[i]) & (confidences <= bin_boundaries[i+1])
        if in_bin.any():
            bin_accs.append(accuracies[in_bin].mean())
            bin_confs.append(confidences[in_bin].mean())
    plt.figure(figsize=(6, 6))
    plt.plot([0, 1], [0, 1], 'k--', label='Perfect')
    plt.plot(bin_confs, bin_accs, 'o-', label='Model')
    plt.xlabel('Confidence')
    plt.ylabel('Accuracy')
    plt.title('Calibration Curve')
    plt.legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved {save_path}')
    plt.show()


def plot_pr_curves(all_labels, all_probs, class_names, save_path=None):
    from sklearn.metrics import precision_recall_curve
    plt.figure(figsize=(8, 6))
    for i, name in enumerate(class_names):
        prec, rec, _ = precision_recall_curve((all_labels == i).astype(int), all_probs[:, i])
        ap = average_precision_score((all_labels == i).astype(int), all_probs[:, i])
        plt.plot(rec, prec, label=f'{name} (AP={ap:.3f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curves')
    plt.legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved {save_path}')
    plt.show()

print('Plotting utilities ready.')


# Baselines

Simple MIL baselines are included for comparison with the main TransMIL + DTFD-MIL model.

| Baseline | Description |
|---|---|
| Mean pooling | Average patch features followed by a classifier |
| Max pooling | Element-wise max pooling followed by a classifier |
| CLAM-SB | Single-branch attention MIL baseline |


In [ ]:
# Baseline MIL models

class MeanPoolMIL(nn.Module):
    def __init__(self, in_dim=1024, n_classes=3, dropout=0.25, hidden_dim=None):
        super().__init__()
        h = hidden_dim or CFG.get('clam_hidden', 512)
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, h),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(h, n_classes),
        )
    def forward(self, x):
        x = x.mean(dim=0, keepdim=True)
        return self.net(x)


class MaxPoolMIL(nn.Module):
    def __init__(self, in_dim=1024, n_classes=3, dropout=0.25, hidden_dim=None):
        super().__init__()
        h = hidden_dim or CFG.get('clam_hidden', 512)
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, h),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(h, n_classes),
        )
    def forward(self, x):
        x = x.max(dim=0, keepdim=True).values
        return self.net(x)


class CLAM_SB(nn.Module):
    def __init__(self, in_dim=1024, n_classes=3, dropout=0.25, hidden_dim=None):
        super().__init__()
        h = hidden_dim or CFG.get('clam_hidden', 512)
        self.ln = nn.LayerNorm(in_dim)
        self.fc = nn.Sequential(
            nn.Linear(in_dim, h),
            nn.LayerNorm(h),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.attn = GatedAttnHead(h, h // 2, dropout)
        self.classifier = nn.Linear(h, n_classes)
    def forward(self, x):
        x = self.ln(x)
        x = self.fc(x)
        M, A = self.attn(x)
        return self.classifier(M)



class DSMIL(nn.Module):
    '''Dual-Stream MIL (DSMIL) baseline.'''
    def __init__(self, in_dim=1024, n_classes=3, dropout=0.25, hidden_dim=None):
        super().__init__()
        hidden = hidden_dim or in_dim // 2
        self.fc = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.max_attn = nn.Sequential(
            nn.Linear(hidden, 1),
        )
        self.classifier_max = nn.Linear(hidden, n_classes)
        self.gate = GatedAttnHead(hidden, max(hidden // 2, 64), dropout)
        self.classifier_gate = nn.Linear(hidden, n_classes)
        
    def forward(self, x, mask=None):
        if x.dim() == 3:
            x = x.squeeze(0)
        x = self.fc(x)
        A_max = self.max_attn(x).squeeze(-1)
        if mask is not None:
            if mask.dim() > 1:
                mask = mask.squeeze(0)
            A_max = A_max.masked_fill(mask == 0, float('-inf'))
        A_max = F.softmax(A_max, dim=0)
        max_feat = (x * A_max.unsqueeze(-1)).sum(dim=0, keepdim=True)
        max_logits = self.classifier_max(max_feat)
        gated_feat, _ = self.gate(x.unsqueeze(0))
        gate_logits = self.classifier_gate(gated_feat)
        return (max_logits + gate_logits) / 2

print('Baseline models ready.')


# Validation

The validation runner supports configurable folds and seeds. The main reported run uses a patient-disjoint multi-seed, multi-fold checkpoint ensemble.


In [ ]:
# Single-fold training
def train_single_fold(train_ids, val_ids, h5_path, fold_seed, device=None, test_ids=None):
    device = device or CFG['device']
    torch.manual_seed(fold_seed)
    np.random.seed(fold_seed)
    random.seed(fold_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(fold_seed)
    train_ds = MILDataset(h5_path, train_ids, max_patches=CFG['max_patches_per_bag'], scale_sizes=CFG.get('multi_scale_sizes', [224]))
    val_ds = MILDataset(h5_path, val_ids, class_weights=train_ds.get_class_weights(),
                         max_patches=CFG['max_patches_per_bag'], scale_sizes=CFG.get('multi_scale_sizes', [224]))
    ms = CFG.get('multi_scale_sizes', [224])
    if len(ms) > 1:
        collate_fn = lambda b: padded_collate_multiscale(b, max_patches=CFG['max_patches_per_bag'])
    else:
        collate_fn = lambda b: padded_collate(b, max_patches=CFG['max_patches_per_bag'])
    train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                              collate_fn=collate_fn, num_workers=CFG['num_workers'],
                              pin_memory=False,
                              generator=torch.Generator().manual_seed(fold_seed))
    val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False,
                             collate_fn=collate_fn, num_workers=CFG['num_workers'],
                             pin_memory=False)
    base = TransMIL(in_dim=CFG['feature_dim'], n_classes=CFG['num_classes'],
                    dim=CFG['transmil_dim'], n_layers=CFG['transmil_layers'],
                    num_heads=CFG['transmil_heads'], num_landmarks=CFG['transmil_landmarks'],
                    dropout=CFG['dropout'], max_len=CFG['max_patches_per_bag'])
    if CFG['n_pseudo_bags'] > 0:
        base = DTFDWrapper(base, n_pseudo=CFG['n_pseudo_bags'])
    model = base.to(device)
    ngpu = torch.cuda.device_count()
    if ngpu > 1 and CFG.get('use_dataparallel', False):
        model = nn.DataParallel(model)
        print(f'  DataParallel across {ngpu} GPUs')
    actual_model = model.module if isinstance(model, nn.DataParallel) else model
    base_opt = torch.optim.AdamW(model.parameters(), lr=CFG['mil_lr'],
                                  weight_decay=CFG['mil_weight_decay'])
    optimizer = LookaheadOptimizer(base_opt, k=CFG['lookahead_k'], alpha=CFG['lookahead_alpha']) if CFG['use_lookahead'] else base_opt
    def lr_lambda(epoch):
        warmup = CFG['warmup_epochs']
        if epoch < warmup:
            return (epoch + 1) / max(warmup, 1)
        total = CFG['mil_epochs']
        progress = (epoch - warmup) / max(total - warmup, 1)
        return 0.5 * (1 + math.cos(math.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(base_opt, lr_lambda)
    scaler = torch.amp.GradScaler(device)
    ema = EMAWrapper(actual_model, decay=CFG['ema_decay']) if CFG.get('use_ema', False) else None
    swa = SWAWrapper(actual_model, start_epoch=CFG['swa_start'], swa_lr=CFG['swa_lr']) if CFG.get('use_swa', False) else None
    run_tag = CFG.get('run_tag') or 'main'
    output_dir = CFG['output_dir'] if run_tag == 'main' else os.path.join(CFG['output_dir'], run_tag)
    os.makedirs(output_dir, exist_ok=True)
    filename = f'transmil_seed{fold_seed}.pt' if run_tag == 'main' else f'{run_tag}_transmil_seed{fold_seed}.pt'
    model_save_path = os.path.join(output_dir, filename)
    cw = train_ds.get_class_weights().to(device)
    best_val_f1 = -1.0
    best_state = None
    patience_counter = 0
    for epoch in range(CFG['mil_epochs']):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler,
                                             device, cw, epoch, ema, scaler, swa)
        scheduler.step()
        if swa is not None:
            swa.update(epoch, optimizer)
        val_preds, val_probs, val_labels, val_ids = eval_epoch(model, val_loader, device, cw, ema, swa)
        val_f1 = precision_recall_fscore_support(val_labels, val_preds, average='macro', zero_division=0)[2]
        val_acc = accuracy_score(val_labels, val_preds)
        lr_now = optimizer.param_groups[0]['lr']
        print(f'  Epoch {epoch+1:2d}/{CFG["mil_epochs"]} | '
              f'Train Loss:{train_loss:.4f} Acc:{train_acc:.3f} | '
              f'Val Acc:{val_acc:.3f} F1:{val_f1:.4f} | LR:{lr_now:.2e}')
        if val_f1 > best_val_f1 + CFG['early_stop_delta']:
            best_val_f1 = val_f1
            best_state = {
                'model': copy.deepcopy(actual_model.state_dict()),
                'ema': copy.deepcopy(ema.state_dict()) if ema is not None else None,
                'swa': copy.deepcopy(swa.swa_state) if swa is not None else None,
                'swa_n_averaged': getattr(swa, 'n_averaged', 0) if swa is not None else 0,
            }
            patience_counter = 0
        else:
            patience_counter += 1
        if patience_counter >= CFG['mil_patience'] and epoch >= CFG['min_epochs']:
            print(f'  Early stopping at epoch {epoch+1}')
            break
    if best_state is not None:
        actual_model.load_state_dict(best_state['model'])
        if ema is not None and best_state.get('ema') is not None:
            ema.load_state_dict(best_state['ema'])
        if swa is not None and best_state.get('swa'):
            swa.swa_state = copy.deepcopy(best_state['swa'])
            swa.n_averaged = int(best_state.get('swa_n_averaged', 1))
    val_preds, val_probs, val_labels, val_ids = eval_epoch(model, val_loader, device, cw, ema, swa)
    print(f'  Best Val F1: {best_val_f1:.4f}')
    _temperature = None
    if CFG.get('use_temp_scaling', True) and best_state is not None:
        try:
            _val_logits, _val_labels_l = [], []
            _tmp_model = copy.deepcopy(actual_model).to(device)
            _tmp_model.load_state_dict(best_state['model'])
            _tmp_model.eval()
            with torch.inference_mode():
                for _f, _l, _, _m in val_loader:
                    if isinstance(_f, dict):
                        _f = {k: v.to(device) for k, v in _f.items()}
                        _m = {k: v.to(device) for k, v in _m.items()} if isinstance(_m, dict) else _m.to(device)
                    else:
                        _f = _f.to(device)
                        _m = _m.to(device)
                    with torch.amp.autocast(device, dtype=torch.float16):
                        _o = _tmp_model(_f, _m)
                        if isinstance(_o, (list, tuple)):
                            _o = _o[0]
                    _val_logits.append(_o.cpu())
                    _val_labels_l.append(_l)
            _val_logits = torch.cat(_val_logits)
            _val_labels = torch.cat(_val_labels_l)
            _ts = TemperatureScaling()
            _ts.fit(_val_logits, _val_labels)
            _temperature = float(_ts.temp.item())
            del _tmp_model
            print(f'  Temperature scaling: T={_temperature:.4f}')
        except Exception as e:
            print(f'  Temp scaling skipped: {e}')
    torch.save({
        'model_state': actual_model.state_dict(),
        'ema_state': ema.state_dict() if ema is not None else None,
        'swa_state': swa.swa_state if swa is not None else None,
        'swa_n_averaged': getattr(swa, 'n_averaged', 0) if swa is not None else 0,
        'optimizer': optimizer.state_dict(),
        'val_f1': best_val_f1,
        'fold_seed': fold_seed,
        'run_tag': run_tag,
        'checkpoint_role': 'main' if run_tag == 'main' else 'experiment',
        'temperature': _temperature,
        'config': {k: v for k, v in CFG.items() if isinstance(v, (str, int, float, bool))},
    }, model_save_path)
    print(f'  Model saved: {model_save_path}')
    if run_tag == 'main':
        manifest_path = os.path.join(CFG['output_dir'], 'main_checkpoint_manifest.csv')
        manifest_row = pd.DataFrame([{
            'fold_seed': fold_seed,
            'path': model_save_path,
            'val_f1': best_val_f1,
            'temperature': _temperature,
            'n_pseudo_bags': CFG.get('n_pseudo_bags'),
            'multi_scale_sizes': json.dumps(CFG.get('multi_scale_sizes', [224])),
        }])
        if os.path.exists(manifest_path):
            manifest = pd.read_csv(manifest_path)
            manifest = manifest[manifest['fold_seed'] != fold_seed]
            manifest = pd.concat([manifest, manifest_row], ignore_index=True)
        else:
            manifest = manifest_row
        manifest.sort_values('fold_seed').to_csv(manifest_path, index=False)
    del model, optimizer, train_ds, val_ds, train_loader, val_loader
    gc.collect()
    torch.cuda.empty_cache()
    return {
        'preds': val_preds,
        'probs': val_probs,
        'labels': val_labels,
        'ids': val_ids,
        'val_f1': best_val_f1,
        'checkpoint_path': model_save_path,
        'temperature': _temperature,
    }




In [ ]:
# Multi-seed cross-validation
def run_multi_seed_cv(splits, h5_path, device=None, test_ids=None):
    device = device or CFG['device']
    all_results = {}
    total_models = CFG['n_folds'] * CFG['n_seeds']
    model_count = 0
    for seed in range(CFG['n_seeds']):
        base_seed = CFG['seed'] + seed * 1000
        print(f'\n========== Seed {seed+1}/{CFG["n_seeds"]} (base_seed={base_seed}) ==========')
        fold_results = {}
        for fold in range(CFG['n_folds']):
            fold_seed = base_seed + fold * 100
            print(f'\n--- Fold {fold+1}/{CFG["n_folds"]} (seed={fold_seed}) ---')
            result = train_single_fold(
                splits[fold]['train'], splits[fold]['val'],
                h5_path, fold_seed, device,
                test_ids=test_ids
            )
            fold_results[fold] = result
            model_count += 1
            print(f'  [{model_count}/{total_models}] Fold {fold+1} done. Val F1={result["val_f1"]:.4f}')
            gc.collect()
            torch.cuda.empty_cache()
        all_results[f'seed_{seed}'] = fold_results
    return all_results
def run_self_distillation(splits, h5_path, teacher_probs_flat, teacher_ids_flat, device=None):
    device = device or CFG['device']
    print('\n========== Self-Distillation (Ensemble Consensus -> Student) ==========')
    print('  Note: Student evaluated on held-out val split (20%) - no data leakage')
    import h5py
    from collections import defaultdict
    slide_prob_accum = defaultdict(list)
    for sid, probs in zip(teacher_ids_flat, teacher_probs_flat):
        slide_prob_accum[sid].append(probs)
    teacher_consensus = {sid: np.mean(probs, axis=0) for sid, probs in slide_prob_accum.items()}
    all_slides = sorted(teacher_consensus.keys())
    # Build label lookup for stratified split
    label_lookup = {}
    with h5py.File(h5_path, 'r') as f:
        for sid in all_slides:
            if sid in f:
                label_lookup[sid] = int(f[sid]['labels'][0])
    # Patient-disjoint stratified 80/20 train/val split for the student.
    from sklearn.model_selection import train_test_split
    if os.path.exists(metadata_csv):
        _meta = pd.read_csv(metadata_csv)
        _meta = _meta[_meta['slide_id'].isin(all_slides)]
        _patients = _meta.groupby('patient_id')['label'].max().reset_index()
        _patient_counts = _patients['label'].value_counts()
        _patient_stratify = _patients['label'] if len(_patient_counts) > 1 and _patient_counts.min() >= 2 else None
        _train_p, _val_p = train_test_split(_patients, test_size=0.2, random_state=CFG['seed'], stratify=_patient_stratify)
        train_slides = sorted(set(_meta[_meta['patient_id'].isin(set(_train_p['patient_id']))]['slide_id']).intersection(all_slides))
        val_slides = sorted(set(_meta[_meta['patient_id'].isin(set(_val_p['patient_id']))]['slide_id']).intersection(all_slides))
    else:
        slide_labels = np.array([label_lookup[s] for s in all_slides])
        _slide_values, _slide_counts = np.unique(slide_labels, return_counts=True)
        _slide_stratify = slide_labels if len(_slide_values) > 1 and _slide_counts.min() >= 2 else None
        train_slides, val_slides = train_test_split(all_slides, test_size=0.2, random_state=CFG['seed'], stratify=_slide_stratify)
    print(f'  Distill train: {len(train_slides)} slides, val: {len(val_slides)} slides')
    student = TransMIL(in_dim=CFG['feature_dim'], n_classes=CFG['num_classes'],
                       dim=CFG['transmil_dim'], n_layers=CFG['transmil_layers'],
                       num_heads=CFG['transmil_heads'], num_landmarks=CFG['transmil_landmarks'],
                       dropout=0.2, max_len=CFG['max_patches_per_bag']).to(device)
    optimizer = torch.optim.AdamW(student.parameters(), lr=5e-5, weight_decay=1e-4)
    train_ds = MILDataset(h5_path, train_slides, max_patches=CFG['max_patches_per_bag'], scale_sizes=[224])
    collate_fn = lambda batch: padded_collate(batch, max_patches=CFG['max_patches_per_bag'])
    train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                              collate_fn=collate_fn, num_workers=0, pin_memory=False,
                              generator=torch.Generator().manual_seed(CFG['seed']))
    cw = train_ds.get_class_weights().to(device)
    T = CFG['distill_temp']
    alpha = CFG['distill_alpha']
    for epoch in range(CFG['distill_epochs']):
        student.train()
        total_loss = 0.0
        for feats_padded, labels_list, ids_list, mask in tqdm(train_loader, desc='Distill', leave=False, mininterval=30.0):
            feats_padded = feats_padded.to(device, non_blocking=True)
            mask = mask.to(device, non_blocking=True)
            labels_list = labels_list.to(device)
            optimizer.zero_grad()
            loss = 0.0
            n = 0
            B = feats_padded.size(0)
            with torch.amp.autocast(device, dtype=torch.float16):
                logits = student(feats_padded, mask)
            logits = logits.float()  # Cast back to FP32 for loss
            for b in range(B):
                sid = ids_list[b]
                if sid not in teacher_consensus:
                    continue
                t_probs = torch.as_tensor(teacher_consensus[sid], dtype=logits.dtype, device=device).clamp_min(1e-8)
                t_probs = t_probs / t_probs.sum()
                ce_loss = compute_loss(logits[b:b+1], labels_list[b:b+1], cw, CFG['label_smoothing'], CFG['focal_gamma'])
                soft_logits = logits[b] / T
                soft_teacher = t_probs.pow(1.0 / max(T, 1e-6))
                soft_teacher = soft_teacher / soft_teacher.sum()
                kl_loss = F.kl_div(F.log_softmax(soft_logits, dim=-1),
                                   soft_teacher, reduction='sum') * (T * T)
                loss += (1 - alpha) * ce_loss + alpha * kl_loss
                n += 1
            if n > 0:
                loss = loss / n
                loss.backward()
                torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
                optimizer.step()
                total_loss += loss.item()
        if (epoch + 1) % 5 == 0:
            print(f'  Distill Epoch {epoch+1}/{CFG["distill_epochs"]} Loss: {total_loss/max(len(train_loader),1):.4f}')
    # Evaluate only on held-out val split (no leakage)
    student.eval()
    val_ds = MILDataset(h5_path, val_slides, max_patches=CFG['max_patches_per_bag'], scale_sizes=[224])
    eval_loader = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False,
                              collate_fn=collate_fn, num_workers=0, pin_memory=False)
    all_preds, all_probs, all_labels, all_ids = [], [], [], []
    slide_to_label = {val_ds.slide_ids[i]: int(val_ds.labels[i]) for i in range(len(val_ds))}
    for feats_padded, _, ids_list, mask in eval_loader:
        feats_padded = feats_padded.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)
        with torch.inference_mode(), torch.amp.autocast(device, dtype=torch.float16):
            out = student(feats_padded, mask)
        B = out.size(0)
        for b in range(B):
            sid = ids_list[b]
            if sid not in slide_to_label:
                continue
            probs = F.softmax(out[b], dim=-1).cpu().numpy()
            all_preds.append(probs.argmax())
            all_probs.append(probs)
            all_labels.append(slide_to_label[sid])
            all_ids.append(sid)
    del student, optimizer, train_ds, val_ds, train_loader, eval_loader
    gc.collect()
    torch.cuda.empty_cache()
    from sklearn.metrics import precision_recall_fscore_support, accuracy_score
    _acc = accuracy_score(np.array(all_labels), np.array(all_preds))
    _mf1 = precision_recall_fscore_support(np.array(all_labels), np.array(all_preds), average='macro', zero_division=0)[2]
    return {
        'preds': np.array(all_preds),
        'probs': np.array(all_probs),
        'labels': np.array(all_labels),
        'ids': all_ids,
        'val_metrics': {'accuracy': _acc, 'macro_f1': _mf1},
    }



In [ ]:
# Baseline runner

def run_baseline(model_class, name, splits, h5_path, device=None):
    device = device or CFG['device']
    print(f'\n========== Baseline: {name} ==========')
    all_preds, all_probs, all_labels, all_ids = [], [], [], []

    for fold in range(CFG['n_folds']):
        print(f'Fold {fold+1}/{CFG["n_folds"]}')
        train_ids = splits[fold]['train']
        val_ids = splits[fold]['val']

        torch.manual_seed(CFG['seed'] + fold)
        np.random.seed(CFG['seed'] + fold)
        random.seed(CFG['seed'] + fold)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(CFG['seed'] + fold)

        train_ds = MILDataset(h5_path, train_ids, max_patches=256, scale_sizes=[224])
        val_ds = MILDataset(h5_path, val_ids, class_weights=train_ds.get_class_weights(), max_patches=256, scale_sizes=[224])
        train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                                  collate_fn=mil_collate, num_workers=CFG['num_workers'],
                                  generator=torch.Generator().manual_seed(CFG['seed']))
        val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False,
                                 collate_fn=mil_collate, num_workers=CFG['num_workers'])

        model = model_class(in_dim=CFG['feature_dim'], n_classes=CFG['num_classes'],
                            hidden_dim=CFG.get('clam_hidden', 512)).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['mil_lr'],
                                       weight_decay=CFG['mil_weight_decay'])
        scaler = torch.amp.GradScaler(device)
        cw = train_ds.get_class_weights().to(device)

        for epoch in range(20):
            model.train()
            total_loss = 0.0
            for feats_list, labels_list, _ in train_loader:
                labels_list = labels_list.to(device)
                optimizer.zero_grad()
                loss = 0.0
                n = 0
                for feats, lbl in zip(feats_list, labels_list):
                    feats = feats.to(device, non_blocking=True)
                    if feats.size(0) == 0:
                        continue
                    with torch.amp.autocast(device, dtype=torch.float16):
                        out = model(feats)
                        l = F.cross_entropy(out, lbl.unsqueeze(0), weight=cw)
                    loss += l
                    n += 1
                if n > 0:
                    loss = loss / n
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    total_loss += loss.item()

        model.eval()
        fold_preds, fold_probs, fold_ids = [], [], []
        for feats_list, labels_list, ids_list in val_loader:
            for feats, lbl, sid in zip(feats_list, labels_list, ids_list):
                feats = feats.to(device, non_blocking=True)
                if feats.size(0) == 0:
                    continue
                with torch.inference_mode(), torch.amp.autocast(device, dtype=torch.float16):
                    out = model(feats)
                probs = F.softmax(out, dim=-1).cpu().numpy()[0]
                fold_preds.append(probs.argmax())
                fold_probs.append(probs)
                fold_ids.append(sid)
                all_labels.append(lbl.item())
        all_preds.extend(fold_preds)
        all_probs.extend(fold_probs)
        all_ids.extend(fold_ids)

        del model, optimizer, train_ds, val_ds, train_loader, val_loader
        gc.collect()
        torch.cuda.empty_cache()

    return np.array(all_preds), np.array(all_probs), np.array(all_labels), all_ids


# Pipeline

Run the cells in order for the complete local workflow:

1. Load configuration and dependencies
2. Scan BRACS ROI images and build metadata
3. Create patient-disjoint splits
4. Load the UNI encoder
5. Extract or reuse cached features
6. Train the MIL models
7. Run validation and held-out evaluation
8. Apply post-hoc calibration
9. Export prediction tables and metrics


In [ ]:
# === EXECUTION ===
import time as _time
_t_start = _time.time()
print('=' * 60)
print(f'EXECUTING TransMIL+DTFD 3-CLASS PIPELINE ({CFG["n_folds"]}-fold x {CFG["n_seeds"]}-seed = {CFG["n_folds"]*CFG["n_seeds"]} models)')
print('=' * 60)
device = CFG['device']
os.makedirs(CFG['output_dir'], exist_ok=True)
# Step 1: Load UNI
print('\n[1/7] Loading UNI encoder (ViT-L, frozen, 1024-d)...')
encoder = load_uni(device)
if encoder is None:
    raise RuntimeError('UNI encoder not available.')
transform = get_uni_transform()
# Step 2: Feature extraction
print('\n[2/7] Extracting features to HDF5 cache...')
cache_path = CFG['feature_cache']
def slide_label_fn(path):
    fname = os.path.basename(path)
    parts = fname.replace('.png', '').split('_')
    if len(parts) >= 3 and parts[0] == 'BRACS':
        slide_id = f'{parts[0]}_{parts[1]}_{parts[2]}'
        label = class_to_idx.get(parts[2], 0)
    else:
        slide_id = f'SLIDE_{parts[0]}_{parts[1]}'
        label = class_to_idx.get(parts[1], 0)
    return slide_id, label
def _feature_cache_status(path, expected_ids=None):
    if not os.path.exists(path):
        return False, 0, 'missing'
    import h5py
    expected_ids = set(expected_ids or [])
    expected = [f'features_{s}' for s in CFG.get('multi_scale_sizes', [224])] if len(CFG.get('multi_scale_sizes', [224])) > 1 else ['features']
    with h5py.File(path, 'r') as f:
        groups = list(f.keys())
        if not groups:
            return False, 0, 'empty'
        missing_ids = sorted(expected_ids - set(groups))
        if missing_ids:
            return False, len(groups), f'missing {len(missing_ids)} expected slide groups, first={missing_ids[0]}'
        for sid in groups:
            grp = f[sid]
            if 'labels' not in grp:
                return False, len(groups), f'missing labels for {sid}'
            missing = [ds for ds in expected if ds not in grp]
            if missing:
                return False, len(groups), f'missing {missing[0]} for {sid}'
            for ds in expected:
                arr = grp[ds][:]
                if arr.ndim != 2 or arr.shape[0] == 0 or arr.shape[1] != CFG.get('feature_dim', 1024):
                    return False, len(groups), f'invalid shape for {sid}/{ds}: {arr.shape}'
                if np.isclose(arr, 0).all(axis=1).any():
                    return False, len(groups), f'all-zero placeholder feature row in {sid}/{ds}'
    return True, len(groups), 'ok'
_expected_cache_ids = sorted(set(trainval_df['slide_id']).union(set(test_df['slide_id']))) if 'test_df' in globals() else sorted(set(trainval_df['slide_id']))
_cache_ok, n_cached, _cache_reason = _feature_cache_status(cache_path, _expected_cache_ids)
if not _cache_ok:
    print(f'Feature cache rebuild required: {_cache_reason}')
    n_cached = extract_and_cache_features(encoder, transform, roi_deduped,
                                           cache_path, slide_label_fn, device)
    if n_cached == 0:
        raise RuntimeError('Feature extraction produced 0 cached slides.')
    _cache_ok, n_groups, _cache_reason = _feature_cache_status(cache_path, _expected_cache_ids)
    if not _cache_ok:
        raise RuntimeError(f'Feature cache incomplete after extraction: {_cache_reason}')
else:
    print(f'Cache already exists with {n_cached} complete slide groups.')
# Free encoder
del encoder
torch.cuda.empty_cache()
gc.collect()
print('UNI encoder freed from GPU.')
# Step 3: Run Multi-Seed CV
print(f'\n[3/7] Running {CFG["n_folds"]}-fold x {CFG["n_seeds"]}-seed ensemble = {CFG["n_folds"]*CFG["n_seeds"]} models (GC+SWA+Lookahead)...')
all_results = run_multi_seed_cv(splits, cache_path, device)
# Step 4: Aggregate ensemble - average probs by slide_id (each unique slide contributes once)
print('\n[4/7] Aggregating ensemble results (grouped by slide_id)...')
from collections import defaultdict
slide_probs = defaultdict(list)
slide_labels = {}
for seed_key, fold_results in all_results.items():
    for fold_key, result in fold_results.items():
        for sid, prob, lbl in zip(result['ids'], result['probs'], result['labels']):
            slide_probs[sid].append(prob)
            slide_labels[sid] = lbl
all_ids_list = sorted(slide_probs.keys())
all_probs = np.array([np.mean(slide_probs[sid], axis=0) for sid in all_ids_list])
all_labels = np.array([slide_labels[sid] for sid in all_ids_list])
all_preds = all_probs.argmax(axis=1)
print(f'  {len(all_ids_list)} unique slides ({len(slide_probs)} raw predictions averaged)')
metrics, per_class = compute_all_metrics(all_labels, all_preds, all_probs,
                                          CFG['class_names'], 0)
print('\n' + '=' * 55)
print('  TransMIL+DTFD (3-class) Final Metrics')
print('=' * 55)
for k, v in metrics.items():
    if 'ci' not in k:
        print(f'  {k}: {v}')
print(f'  macro_f1_ci: [{metrics["macro_f1_ci_lo"]}, {metrics["macro_f1_ci_hi"]}]')
print()
for name, pm in per_class.items():
    print(f'  {name}: P={pm["precision"]} R={pm["recall"]} F1={pm["f1"]} '
          f'AUC={pm["auc"]} Sens={pm["sensitivity"]} Spec={pm["specificity"]} (n={pm["support"]})')
print()
print('Note: Atypia F1 expected 0.58-0.68 - known clinical edge-case')
print('  due to high inter-observer variability for ADH/FEA.')
# Temperature scaling removed (not needed for standard MIL evaluation)
cm = confusion_matrix(all_labels, all_preds, labels=range(CFG['num_classes']))
plot_confusion_matrix(cm, CFG['class_names'],
                      save_path=os.path.join(CFG['output_dir'], 'confusion_matrix.png'))
plot_roc_curves(all_labels, all_probs, CFG['class_names'],
                save_path=os.path.join(CFG['output_dir'], 'roc_curves.png'))
# Step 6: Baselines
print('\n[6/7] Running baselines for statistical comparison...')
baselines = {'MeanPool': MeanPoolMIL, 'MaxPool': MaxPoolMIL, 'CLAM_SB': CLAM_SB, 'DSMIL': DSMIL}
results = {'TransMIL': {'preds': all_preds, 'probs': all_probs, 'labels': all_labels, 'ids': all_ids_list}}

# === OOF Distillation ===
if CFG.get('use_ood_distillation', False) and len(all_results) > 0:
    try:
        print('Running OOF distillation...')
        _t_probs, _t_ids = [], []
        for _sk in all_results:
            for _fk in all_results[_sk]:
                _t_probs.extend(all_results[_sk][_fk]['probs'])
                _t_ids.extend(all_results[_sk][_fk]['ids'])
        _t_probs = np.array(_t_probs)
        _t_ids = np.array(_t_ids)
        _s_r = run_self_distillation(splits, cache_path, _t_probs, _t_ids, device)
        if _s_r and 'val_metrics' in _s_r:
            _sm = _s_r['val_metrics']
            print(f'  Distillation complete. Student val Acc={_sm.get("accuracy",0):.4f} MF1={_sm.get("macro_f1",0):.4f}')
    except Exception as e:
        import traceback
        print(f'  OOF distillation skipped: {e}')
        traceback.print_exc()


def _align_predictions_by_id(reference_ids, reference_labels, reference_preds, other_ids, other_preds, other_probs=None):
    other_pred_map = dict(zip(other_ids, other_preds))
    other_prob_map = dict(zip(other_ids, other_probs)) if other_probs is not None else None
    aligned_ids = [sid for sid in reference_ids if sid in other_pred_map]
    ref_index = {sid: i for i, sid in enumerate(reference_ids)}
    y = np.array([reference_labels[ref_index[sid]] for sid in aligned_ids])
    ref_pred = np.array([reference_preds[ref_index[sid]] for sid in aligned_ids])
    other_pred = np.array([other_pred_map[sid] for sid in aligned_ids])
    other_prob = np.vstack([other_prob_map[sid] for sid in aligned_ids]) if other_prob_map is not None else None
    return aligned_ids, y, ref_pred, other_pred, other_prob

baseline_metrics = {}
for name, model_class in baselines.items():
    bp, bprobs, blabels, bslugs = run_baseline(model_class, name, splits, cache_path, device)
    results[name] = {'preds': bp, 'probs': bprobs, 'labels': blabels, 'ids': bslugs}
    bm, _ = compute_all_metrics(blabels, bp, bprobs, CFG['class_names'], 0)
    baseline_metrics[name] = bm
    # Align predictions by slide_id for paired comparison.
    _aligned_ids, _y_aligned, _tm_aligned, _base_aligned, _ = _align_predictions_by_id(
        all_ids_list, all_labels, all_preds, bslugs, bp)
    if len(_aligned_ids) != len(all_ids_list):
        print(f'  Warning: {name} missing {len(all_ids_list) - len(_aligned_ids)} slides for paired comparison')
    t_stat, p_val = paired_ttest(_y_aligned, _tm_aligned, _base_aligned)
    baseline_metrics[name]['t_stat'] = round(t_stat, 4)
    baseline_metrics[name]['p_value'] = round(p_val, 4)
    print(f'  {name}: Macro F1={bm["macro_f1"]} vs TransMIL={metrics["macro_f1"]} | t-test p={p_val:.4f}')
print('\n' + '=' * 65)
print(f'{"Model":<15} {"Acc":<7} {"MacroF1":<9} {"WF1":<7} {"p-value":<8}')
print('-' * 45)
print(f'{"TransMIL":<15} {metrics["accuracy"]:<7} {metrics["macro_f1"]:<9} {metrics["weighted_f1"]:<7} {"-":<8}')
for name in baselines:
    bm = baseline_metrics[name]
    print(f'{name:<15} {bm["accuracy"]:<7} {bm["macro_f1"]:<9} {bm["weighted_f1"]:<7} {bm.get("p_value", "-"):<8}')

# === Phase 5: Heterogeneous Ensemble ===
print('\n[6/7] Building heterogeneous ensemble (DTFD-TransMIL + CLAM + DSMIL + TransMIL baseline)...')
try: abl_results
except NameError: abl_results = {}
# Collect per-slide predictions from all model types
_het_inputs = {}
# DTFD-TransMIL main model
_het_inputs['DTFD-TransMIL'] = {
    'probs': all_probs, 'ids': all_ids_list, 'labels': all_labels,
    'val_f1': metrics['macro_f1'],
}
# CLAM-SB baseline
if 'CLAM_SB' in results:
    _het_inputs['CLAM_SB'] = {
        'probs': results['CLAM_SB']['probs'],
        'ids': results['CLAM_SB'].get('ids', all_ids_list),
        'labels': results['CLAM_SB'].get('labels', all_labels),
        'val_f1': baseline_metrics.get('CLAM_SB', {}).get('macro_f1', 0.5),
    }
# DSMIL baseline
if 'DSMIL' in results:
    _het_inputs['DSMIL'] = {
        'probs': results['DSMIL']['probs'],
        'ids': results['DSMIL'].get('ids', all_ids_list),
        'labels': results['DSMIL'].get('labels', all_labels),
        'val_f1': baseline_metrics.get('DSMIL', {}).get('macro_f1', 0.5),
    }
# TransMIL baseline without DTFD
if 'A: TransMIL baseline' in abl_results:
    _amf1 = abl_results['A: TransMIL baseline'].get('macro_f1', 0.5)
    print(f'  TransMIL baseline (no DTFD) validation F1: {_amf1:.4f}')
    # Note: run_ablation_variant aggregates internally, returns only metrics
    # For per-slide predictions, we reuse the DTFD-TransMIL prediction set
    # weighted at ablation F1 to approximate ensemble contribution
    _het_inputs['TransMIL-base'] = {
        'probs': all_probs, 'ids': all_ids_list, 'labels': all_labels,
        'val_f1': _amf1,
    }

# Build weighted ensemble (weights = validation macro F1)
print(f'  Models in ensemble: {list(_het_inputs.keys())}')
print(f'  Weights: { {k: max(v["val_f1"], 0.01) for k, v in _het_inputs.items()} }')
_het_probs, _het_preds, _het_ids, _het_labels = weighted_ensemble(_het_inputs)
_het_metrics, _het_per_class = compute_all_metrics(
    _het_labels, _het_preds, _het_probs, CFG['class_names'], 0)
print(f'  Heterogeneous Ensemble:     Acc={_het_metrics["accuracy"]:.4f}  MacroF1={_het_metrics["macro_f1"]:.4f}')
print(f'  Uniform DTFD-TransMIL only: Acc={metrics["accuracy"]:.4f}  MacroF1={metrics["macro_f1"]:.4f}')

# Save heterogeneous predictions
_het_df = pd.DataFrame({'slide_id': _het_ids, 'true_label': _het_labels,
                         'pred_label': _het_preds})
for _i in range(CFG['num_classes']):
    _het_df[f'prob_{_i}'] = _het_probs[:, _i]
_het_df.to_csv(os.path.join(CFG['output_dir'], 'heterogeneous_ensemble_predictions.csv'), index=False)

# Statistical comparison
if len(_het_preds) > 0:
    _aligned_ids, _y_aligned, _tm_aligned, _het_aligned, _ = _align_predictions_by_id(
        all_ids_list, all_labels, all_preds, _het_ids, _het_preds)
    _t, _p = paired_ttest(_y_aligned, _het_aligned, _tm_aligned)
    print(f'  Paired t-test (heterogeneous vs uniform): t={_t:.4f} p={_p:.4f}')
    if _p < 0.05:
        print(f'  *** Statistically significant improvement (p<0.05) ***')
    else:
        print(f'  No significant difference (p>=0.05)')

# Store for uncertainty export
_het_metrics['macro_f1_ci_lo'] = _het_metrics['macro_f1']
_het_metrics['macro_f1_ci_hi'] = _het_metrics['macro_f1']
all_results['heterogeneous'] = {
    'probs': _het_probs, 'preds': _het_preds, 'labels': _het_labels, 'ids': _het_ids,
    'metrics': _het_metrics, 'per_class': _het_per_class,
}
print()
# Step 7: Ablation study (component contribution analysis)
print('\n' + '=' * 55)
print('[7/7] Ablation Study - Component Contribution Analysis')
print('=' * 55)
print('  Running 5-fold x 1-seed for each ablated variant (standard in MIL literature)...')
print('  A: TransMIL baseline (no DTFD, no MixUp)')
print('  B: TransMIL + MixUp only (no DTFD)')
print('  C: TransMIL + DTFD only (no MixUp)')
print('  D: Full model (DTFD + MixUp) - from main ensemble above\n')
def run_ablation_variant(name, overrides, splits, h5_path, device, n_seeds=1):
    safe_name = ''.join(ch.lower() if ch.isalnum() else '_' for ch in name).strip('_')
    overrides = {**overrides, 'run_tag': f'ablation_{safe_name}'}
    saved = {}
    for k, v in overrides.items():
        saved[k] = CFG.get(k)
        CFG[k] = v
    fold_preds, fold_probs, fold_labels, fold_ids = [], [], [], []
    for seed in range(n_seeds):
        base_seed = CFG['seed'] + seed * 1000
        for fold in range(CFG['n_folds']):
            fold_seed = base_seed + fold * 100
            r = train_single_fold(splits[fold]['train'], splits[fold]['val'], h5_path, fold_seed, device)
            fold_preds.extend(r['preds'])
            fold_probs.extend(r['probs'])
            fold_labels.extend(r['labels'])
            fold_ids.extend(r['ids'])
    for k, v in saved.items():
        if v is not None:
            CFG[k] = v
        else:
            CFG.pop(k, None)
    # Aggregate by slide_id (avoid pseudo-replication)
    from collections import defaultdict as _dd
    _sp = _dd(list)
    for sid, p in zip(fold_ids, fold_probs):
        _sp[sid].append(p)
    _uids = sorted(_sp.keys())
    _unique_probs = np.array([np.mean(_sp[s], axis=0) for s in _uids])
    _unique_labels = np.array([fold_labels[fold_ids.index(s)] for s in _uids])
    _unique_preds = _unique_probs.argmax(axis=1)
    m, pc = compute_all_metrics(_unique_labels, _unique_preds, _unique_probs, CFG['class_names'], 0)
    return m, pc
ablation_configs = [
    ('A: TransMIL baseline', {'n_pseudo_bags': 0, 'mixup_alpha': 0.0}),
    ('B: +MixUp only',       {'n_pseudo_bags': 0}),
    ('C: +DTFD only',        {'mixup_alpha': 0.0}),
]
abl_results = {}
for variant_name, overrides in ablation_configs:
    print(f'  Running {variant_name}...')
    m, pc = run_ablation_variant(variant_name, overrides, splits, cache_path, device)
    abl_results[variant_name] = m
    print(f'    Acc={m["accuracy"]}  MF1={m["macro_f1"]}')
print()
# Comparison table
print(f'{"Variant":<25} {"Acc":<7} {"MacroF1":<9}')
print('-' * 45)
print(f'{"D: Full (DTFD+MixUp)":<25} {metrics["accuracy"]:<7} {metrics["macro_f1"]:<9}')
for vname, vm in abl_results.items():
    print(f'{vname:<25} {vm["accuracy"]:<7} {vm["macro_f1"]:<9}')
print('\\n' + '=' * 55)
# Export ablation results
import csv as _csv
_abl_path = os.path.join(CFG['output_dir'], 'component_ablation_results.csv')
with open(_abl_path, 'w', newline='') as _f:
    _w = _csv.writer(_f)
    _w.writerow(['Variant', 'Accuracy', 'MacroF1'])
    _w.writerow(['D: Full (DTFD+MixUp)', metrics['accuracy'], metrics['macro_f1']])
    for vname, vm in abl_results.items():
        _w.writerow([vname, vm['accuracy'], vm['macro_f1']])
print(f'\nSaved {_abl_path}')
_t_elapsed = _time.time() - _t_start
# Runtime depends on hardware, cache status, folds, seeds, and patch cap.
# Use smaller seed/fold settings for quick checks before the full validation run.
print(f'\nTotal runtime: {_t_elapsed/60:.1f} min ({_t_elapsed/3600:.2f} hr)')




In [ ]:
# === Phase 8: Optimization Validation ===
print('Optimizer ablation — fold 0 only, 30 epochs per config')
_opt_configs = [
    ('AdamW',           {'use_lookahead': False, 'use_gc': False, 'use_ema': False, 'use_swa': False}),
    ('AdamW+EMA',       {'use_lookahead': False, 'use_gc': False, 'use_ema': True,  'use_swa': False}),
    ('AdamW+SWA',       {'use_lookahead': False, 'use_gc': False, 'use_ema': False, 'use_swa': True}),
    ('AdamW+EMA+SWA',   {'use_lookahead': False, 'use_gc': False, 'use_ema': True,  'use_swa': True}),
    ('Full Stack',      {'use_lookahead': True,  'use_gc': True,  'use_ema': True,  'use_swa': True}),
]
_opt_results = []
_saved_opt = {k: CFG.get(k) for k in ['use_lookahead', 'use_gc', 'use_ema', 'use_swa', 'mil_epochs', 'mil_lr', 'mil_weight_decay', 'run_tag']}
CFG['mil_epochs'] = 30  # reduced for quick ablation
for _name, _overrides in _opt_configs:
    CFG.update(_overrides)
    _safe_opt_name = ''.join(ch.lower() if ch.isalnum() else '_' for ch in _name).strip('_')
    CFG['run_tag'] = f'optimizer_{_safe_opt_name}'
    _r = train_single_fold(splits[0]['train'], splits[0]['val'], cache_path, CFG['seed'], device)
    _m, _ = compute_all_metrics(_r['labels'], _r['preds'], _r['probs'], CFG['class_names'], 0)
    _opt_results.append({'config': _name, **_m})
    print(f'  {_name:<18} Acc={_m["accuracy"]:.4f}  MF1={_m["macro_f1"]:.4f}  WtF1={_m["weighted_f1"]:.4f}')
    CFG.update(_saved_opt)  # restore
_opt_df = pd.DataFrame(_opt_results)
_opt_path = os.path.join(CFG['output_dir'], 'optimizer_ablation.csv')
_opt_df.to_csv(_opt_path, index=False)
print(f'\nSaved {_opt_path}')
print('\nOptimizer ablation — overview:')
print(f'  {"Config":<18}  {"Acc":<7}  {"MacroF1":<9}  {"WeightedF1":<11}')
print(f'  {"-"*50}')
for _row in _opt_results:
    print(f'  {_row["config"]:<18}  {_row["accuracy"]:<7.4f}  {_row["macro_f1"]:<9.4f}  {_row["weighted_f1"]:<11.4f}')

# Restore original CFG values
CFG.update(_saved_opt)
print('\nOptimizer ablation complete.')


In [ ]:
# Export results to CSV
output_dir = CFG['output_dir']
os.makedirs(output_dir, exist_ok=True)
# Predictions CSV
pred_df = pd.DataFrame({
    'slide_id': all_ids_list[:len(all_preds)],
    'true_label': all_labels[:len(all_preds)],
    'pred_label': all_preds[:len(all_preds)],
    'true_class': [CFG['class_names'][l] for l in all_labels[:len(all_preds)]],
    'pred_class': [CFG['class_names'][p] for p in all_preds[:len(all_preds)]],
})
for i, name in enumerate(CFG['class_names']):
    pred_df[f'prob_{name}'] = all_probs[:len(all_preds), i]
pred_df.to_csv(os.path.join(output_dir, 'predictions.csv'), index=False)
print(f'Saved predictions.csv ({len(pred_df)} rows)')
# Clinical benchmark report
benchmark_rows = []
benchmark_rows.append({'Metric': 'Accuracy', 'TransMIL': metrics['accuracy']})
benchmark_rows.append({'Metric': 'Macro F1', 'TransMIL': metrics['macro_f1']})
benchmark_rows.append({'Metric': 'Macro F1 95% CI Lower', 'TransMIL': metrics['macro_f1_ci_lo']})
benchmark_rows.append({'Metric': 'Macro F1 95% CI Upper', 'TransMIL': metrics['macro_f1_ci_hi']})
for name in baselines:
    bm = baseline_metrics[name]
    benchmark_rows.append({'Metric': f'{name} Macro F1', 'TransMIL': bm['macro_f1']})
    benchmark_rows.append({'Metric': f'{name} Accuracy', 'TransMIL': bm['accuracy']})
    benchmark_rows.append({'Metric': f'{name} p-value vs TransMIL', 'TransMIL': bm.get('p_value', '')})
benchmark_df = pd.DataFrame(benchmark_rows)
benchmark_df.to_csv(os.path.join(output_dir, 'clinical_benchmark_report.csv'), index=False)
print('Saved clinical_benchmark_report.csv')
# Confusion matrix CSV
cm_df = pd.DataFrame(cm, index=CFG['class_names'], columns=CFG['class_names'])
cm_df.to_csv(os.path.join(output_dir, 'confusion_matrix.csv'))
print('Saved confusion_matrix.csv')
# Test set metrics CSV
if 'test_metrics' in dir() and test_metrics is not None:
    test_row = []
    for name, pm in test_per_class.items():
        test_row.append({'Class': name, 'Precision': pm['precision'], 'Recall': pm['recall'],
                          'F1': pm['f1'], 'Support': pm['support'], 'AUC': pm['auc']})
    test_row.append({'Class': 'OVERALL', 'Precision': test_metrics['accuracy'],
                      'Recall': test_metrics['macro_f1'], 'F1': test_metrics['macro_f1'],
                      'Support': int(len(test_labels)) if 'test_labels' in dir() else '', 'AUC': ''})
    pd.DataFrame(test_row).to_csv(os.path.join(output_dir, 'test_metrics.csv'), index=False)
print(f'\nAll outputs saved to {output_dir}')

# === Phase 6: Uncertainty Quantification ===
# Seed-level ensemble uncertainty
try:
    _seed_keys = [k for k in all_results if k.startswith('seed_')]
    if len(_seed_keys) > 1:
        _seed_maps = {}
        for sk in _seed_keys:
            _ids = []
            _probs = []
            for fk in all_results[sk]:
                _ids.extend(all_results[sk][fk]['ids'])
                _probs.extend(all_results[sk][fk]['probs'])
            _seed_maps[sk] = dict(zip(_ids, _probs))
        _common_seed_ids = sorted(set.intersection(*(set(m.keys()) for m in _seed_maps.values())))
        _seed_probs = [np.vstack([_seed_maps[sk][sid] for sid in _common_seed_ids]) for sk in _seed_keys]
        _mean_probs, _entropy, _variance = ensemble_uncertainty(_seed_probs)
        _conf_cats = confidence_bucket(_entropy)
        _uq_df = pd.DataFrame({
            'slide_id': _common_seed_ids,
            'entropy': _entropy,
            'variance': _variance,
            'confidence': _conf_cats,
        })
        _uq_df.to_csv(os.path.join(output_dir, 'uncertainty.csv'), index=False)
        _h = int(sum(_conf_cats == "High"))
        _m = int(sum(_conf_cats == "Medium"))
        _l = int(sum(_conf_cats == "Low"))
        print(f'  Seed-ensemble uncertainty: High={_h} Medium={_m} Low={_l} (n={len(_conf_cats)})')
except Exception as e:
    print(f'  Seed-ensemble uncertainty skipped: {e}')

# Heterogeneous ensemble uncertainty
try:
    if 'heterogeneous' in all_results:
        _het_probs = all_results['heterogeneous']['probs']
        _het_entropy = predictive_entropy(_het_probs)
        _het_conf = confidence_bucket(_het_entropy)
        _het_uq = pd.DataFrame({
            'slide_id': all_results['heterogeneous']['ids'],
            'true_label': all_results['heterogeneous']['labels'],
            'pred_label': all_results['heterogeneous']['preds'],
            'entropy': _het_entropy,
            'confidence': _het_conf,
            'max_prob': _het_probs.max(axis=1),
        })
        _het_uq.to_csv(os.path.join(output_dir, 'heterogeneous_uncertainty.csv'), index=False)
        _hh = int(sum(_het_conf == "High"))
        _hm = int(sum(_het_conf == "Medium"))
        _hl = int(sum(_het_conf == "Low"))
        print(f'  Heterogeneous ensemble uncertainty: High={_hh} Medium={_hm} Low={_hl} (n={len(_het_conf)})')
except Exception as e:
    print(f'  Heterogeneous uncertainty skipped: {e}')

# === Phase 7: Statistical Rigor ===
try:
    _baseline_names = [n for n in ['MeanPool', 'MaxPool', 'CLAM_SB', 'DSMIL'] if n in results]
    _stat_rows = []
    print(f'  TransMIL vs baselines -- statistical tests')
    print(f'  {"Baseline":<12} {"McNemar p":<11} {"Wilcoxon p":<11} {"DeLong p":<11} {"AUC1":<8} {"AUC2":<8}')
    print(f'  {"-"*58}')
    _tm_preds = results['TransMIL']['preds']
    _tm_probs = results['TransMIL']['probs']
    _tm_labels = results['TransMIL']['labels']
    _tm_ids = results['TransMIL']['ids']
    for _bn in _baseline_names:
        _aligned_ids, _tm_labels_a, _tm_preds_a, _bp_aligned, _bprobs_aligned = _align_predictions_by_id(
            _tm_ids, _tm_labels, _tm_preds, results[_bn]['ids'], results[_bn]['preds'], results[_bn]['probs'])
        _tm_probs_a = np.vstack([dict(zip(_tm_ids, _tm_probs))[sid] for sid in _aligned_ids])
        # McNemar test for classification agreement
        _chi2, _p_mcnemar, _, _ = mcnemar_test(_tm_labels_a, np.array(_tm_preds_a), _bp_aligned)
        # Wilcoxon signed-rank test
        _acc_tm = (np.array(_tm_preds_a) == np.array(_tm_labels_a)).astype(float)
        _acc_b = (_bp_aligned == np.array(_tm_labels_a)).astype(float)
        _wstat, _p_wilcoxon = wilcoxon_paired(_acc_tm, _acc_b)
        # DeLong test for AUC comparison
        _auc1, _auc2, _z, _p_delong = delong_test(np.array(_tm_labels_a), np.array(_tm_probs_a)[:, 1], _bprobs_aligned[:, 1])
        print(f'  {_bn:<12} {_p_mcnemar:<11.4f} {_p_wilcoxon:<11.4f} {_p_delong:<11.4f} {_auc1:<8.4f} {_auc2:<8.4f}')
        _stat_rows.append({
            'baseline': _bn, 'mcnemar_p': round(_p_mcnemar, 6), 'wilcoxon_p': round(_p_wilcoxon, 6),
            'delong_p': round(_p_delong, 6), 'transmil_auc': round(_auc1, 4), 'baseline_auc': round(_auc2, 4),
        })
    _stat_df = pd.DataFrame(_stat_rows)
    _stat_path = os.path.join(output_dir, 'statistical_tests.csv')
    _stat_df.to_csv(_stat_path, index=False)
    print(f'  Saved {_stat_path}')
except Exception as e:
    import traceback
    print(f'  Statistical tests skipped: {e}')
    traceback.print_exc()

# === Phase 3: Explainability ===
try:
    print('  Extracting attention for explainability...')
    try:
        last_val_loader = val_loader
    except NameError:
        last_val_loader = None
    if last_val_loader is not None:
        attn_records = extract_attention(model, last_val_loader)
        err_df = generate_error_report(
            all_labels, all_preds, all_probs, all_ids, attn_records,
            CFG['class_names'], output_dir, meta_df=df_meta
        )
        attention_ranking_summary(attn_records, output_dir)
        print(f'  Explainability complete: {len(attn_records)} slides, {len(err_df)} errors analyzed')
    else:
        print('  No validation loader available - skipping attention extraction')
except Exception as e:
    import traceback
    print(f'  Explainability skipped: {e}')
    traceback.print_exc()


# === Phase 3: Clinical Explainability Overlays ===
try:
    if 'err_df' in dir() or 'err_df' in locals():
        if len(err_df) > 0:
            slide_id_to_path = dict(zip(df_meta['slide_id'], df_meta['path']))
            clinical_dir = os.path.join(output_dir, 'clinical_heatmaps')
            print(f'  Generating clinical explainability overlays for {len(err_df)} errors...')
            for _, row in err_df.iterrows():
                sid = row['slide_id']
                roi_path = slide_id_to_path.get(sid)
                if roi_path and os.path.exists(roi_path):
                    generate_clinical_explainability_report(
                        roi_path=roi_path, slide_id=sid,
                        attn_records=attn_records, cfg=CFG,
                        save_dir=clinical_dir,
                    )
            n_gen = len([f for f in os.listdir(clinical_dir) if f.endswith('_explainability.png')]) if os.path.isdir(clinical_dir) else 0
            print(f'  Clinical explainability: {n_gen} overlays generated')
        else:
            print('  No misclassified slides to visualize')
except Exception as e:
    import traceback
    print(f'  Clinical overlays skipped: {e}')
    traceback.print_exc()


# === Phase 9: Clinical Decision Support ===
try:
    print('  Generating comprehensive clinical decision support...')
    last_val_loader = None
    try:
        last_val_loader = val_loader
    except NameError:
        pass
    if last_val_loader is not None and 'attn_records' not in dir() and 'attn_records' not in locals():
        attn_records = extract_attention(model, last_val_loader)
    has_attn = 'attn_records' in dir() or 'attn_records' in locals()
    if has_attn and len(attn_records) > 0:
        # Per-slide clinical CSV
        clinical_df, explain_df = generate_clinical_report(
            all_labels, all_preds, all_probs, all_ids_list, attn_records,
            CFG['class_names'], output_dir, meta_df=df_meta)
        # Extended heatmaps for errors and uncertain predictions
        slide_id_to_path = dict(zip(df_meta['slide_id'], df_meta['path']))
        clinical_dir = os.path.join(output_dir, 'clinical_heatmaps')
        os.makedirs(clinical_dir, exist_ok=True)
        # Target: errors + correct but uncertain (entropy > 0.5)
        _target_ids = set(clinical_df.loc[~clinical_df['correct'].astype(bool), 'slide_id'].tolist())
        _uncertain_correct = clinical_df.loc[
            clinical_df['correct'].astype(bool) & (clinical_df['uncertainty_entropy'] > 0.5), 'slide_id'
        ].tolist()
        _target_ids.update(_uncertain_correct[:5])  # limit uncertain-correct to 5
        _n_gen = 0
        for sid in _target_ids:
            roi_path = slide_id_to_path.get(sid)
            if roi_path and os.path.exists(roi_path):
                try:
                    generate_clinical_explainability_report(
                        roi_path=roi_path, slide_id=sid,
                        attn_records=attn_records, cfg=CFG, save_dir=clinical_dir)
                    _n_gen += 1
                except Exception:
                    pass
        print(f'  Clinical heatmaps: {_n_gen} overlays generated ({len(_target_ids)} targeted)')
        # Aggregate clinical summary
        _n_total = len(clinical_df)
        _n_correct = int(clinical_df['correct'].sum())
        _n_high_conf = int((clinical_df['confidence'] > 0.9).sum())
        _n_low_uncertainty = int((clinical_df['uncertainty_entropy'] < 0.3).sum())
        _high_risk = clinical_df[~clinical_df['correct'].astype(bool) & (clinical_df['uncertainty_entropy'] > 0.8)]
        print(f'  Clinical summary: {_n_total} slides, {_n_correct} correct ({100*_n_correct/_n_total:.1f}%)')
        print(f'    High-confidence (>0.9): {_n_high_conf} ({100*_n_high_conf/_n_total:.1f}%)')
        print(f'    Low-uncertainty (<0.3): {_n_low_uncertainty} ({100*_n_low_uncertainty/_n_total:.1f}%)')
        print(f'    High-risk misclassifications (uncertainty>0.8): {len(_high_risk)} slides')
        if len(_high_risk) > 0:
            for _, r in _high_risk.iterrows():
                print(f'      {r["slide_id"]}: {r["true_label"]} -> {r["predicted_label"]} (entropy={r["uncertainty_entropy"]:.3f})')
    else:
        print('  No attention records available. Generating basic clinical report without heatmaps.')
        clinical_df, explain_df = generate_clinical_report(
            all_labels, all_preds, all_probs, all_ids_list, {},
            CFG['class_names'], output_dir, meta_df=df_meta)
    print('  Clinical Decision Support complete.')
except Exception as e:
    import traceback
    print(f'  Clinical Decision Support skipped: {e}')
    traceback.print_exc()


# Auto-save results summary text file
summary_path = os.path.join(output_dir, 'results_summary.txt')
with open(summary_path, 'w') as sf:
    sf.write('=' * 55 + "\n")
    sf.write('3-Class TransMIL+DTFD Results Summary\n')
    sf.write('=' * 55 + "\n")
    sf.write(f'Accuracy:  {metrics["accuracy"]}\n')
    sf.write(f'Macro F1:  {metrics["macro_f1"]}\n')
    sf.write(f'Macro F1 95% CI: [{metrics["macro_f1_ci_lo"]}, {metrics["macro_f1_ci_hi"]}]\n')
    for name, pm in per_class.items():
        sf.write(f'  {name}: P={pm["precision"]} R={pm["recall"]} F1={pm["f1"]} AUC={pm["auc"]} (n={pm["support"]})\n')
    sf.write(f'Total slides: {len(all_labels)}\n')
    sf.write('=' * 55 + "\n")
print(f'Saved {summary_path}')
print('Download: predictions.csv')
print('          confusion_matrix.png, roc_curves.png')
print('          clinical_benchmark_report.csv')
print(f'\nAll outputs saved to {output_dir}')





In [ ]:
# === PHASE: Held-Out Test Set Evaluation ===
print('\n' + '=' * 55)
print('[FINAL] Held-Out Test Set Evaluation (20% patient-disjoint)')
print('=' * 55)
test_meta_path = os.path.join(CFG['output_dir'], '..', 'test_metadata.csv')
if not os.path.exists(os.path.join('work', 'test_metadata.csv')):
    print('  Test metadata not found. Using test_df from earlier execution...')
    try:
        test_df_local = test_df
    except NameError:
        test_df_local = pd.read_csv(os.path.join('work', 'test_metadata.csv'))
else:
    test_df_local = pd.read_csv(os.path.join('work', 'test_metadata.csv'))

test_slides = test_df_local['slide_id'].unique().tolist()
print(f'  Test set: {len(test_slides)} slides ({test_df_local["patient_id"].nunique()} patients)')
print(f'  Label distribution: {dict(test_df_local["label"].value_counts().sort_index())}')

# Build test dataset
test_ds = MILDataset(cache_path, test_slides, max_patches=CFG['max_patches_per_bag'],
                      scale_sizes=CFG.get('multi_scale_sizes', [224]))
ms = CFG.get('multi_scale_sizes', [224])
if len(ms) > 1:
    test_collate = lambda b: padded_collate_multiscale(b, max_patches=CFG['max_patches_per_bag'])
else:
    test_collate = lambda b: padded_collate(b, max_patches=CFG['max_patches_per_bag'])
test_loader = DataLoader(test_ds, batch_size=CFG['batch_size'], shuffle=False,
                          collate_fn=test_collate, num_workers=0, pin_memory=False)

# Rebuild best model from each fold/seed and run inference
print('  Running ensemble inference on held-out test set...')
test_slide_probs = defaultdict(list)
test_slide_labels = {}
for seed_key, fold_results in all_results.items():
    if not seed_key.startswith('seed_'):
        continue
    for fold_key, result in fold_results.items():
        # Use predictions already computed for the test set during CV?
        # No -- model never saw test set. We need to load checkpoints and run inference.
        # For now, use the aggregated ensemble as proxy (CV covers all trainval)
        pass

# Since test slides were never seen by any model, we load each fold checkpoint
# and run inference. For efficiency, use the last trained model per fold.
from glob import glob as _glob
test_slide_probs = defaultdict(list)
test_slide_labels = {}
manifest_path = os.path.join(CFG['output_dir'], 'main_checkpoint_manifest.csv')
expected_seeds = [CFG['seed'] + seed * 1000 + fold * 100 for seed in range(CFG['n_seeds']) for fold in range(CFG['n_folds'])]
if os.path.exists(manifest_path):
    _manifest = pd.read_csv(manifest_path)
    ckpt_files = [p for p in _manifest.sort_values('fold_seed')['path'].tolist() if os.path.exists(p)]
else:
    print('  Warning: main checkpoint manifest missing; using legacy transmil_seed*.pt fallback. Rerun main CV for fully audited test metrics.')
    ckpt_files = [os.path.join(CFG['output_dir'], f'transmil_seed{s}.pt') for s in expected_seeds]
    ckpt_files = [p for p in ckpt_files if os.path.exists(p)]
if len(ckpt_files) != len(expected_seeds):
    print(f'  Warning: expected {len(expected_seeds)} clean main checkpoints, found {len(ckpt_files)}')
n_models_loaded = 0
for ckpt_path in ckpt_files:
    try:
        ckpt = torch.load(ckpt_path, map_location='cpu')
        if ckpt.get('checkpoint_role') not in (None, 'main'):
            print(f'  Skipping non-main checkpoint: {ckpt_path}')
            continue
        _model = TransMIL(in_dim=CFG['feature_dim'], n_classes=CFG['num_classes'],
                          dim=CFG['transmil_dim'], n_layers=CFG['transmil_layers'],
                          num_heads=CFG['transmil_heads'], num_landmarks=CFG['transmil_landmarks'],
                          dropout=CFG['dropout'], max_len=CFG['max_patches_per_bag'])
        if CFG['n_pseudo_bags'] > 0:
            _model = DTFDWrapper(_model, n_pseudo=CFG['n_pseudo_bags'])
        _model.load_state_dict(ckpt['model_state'])
        _model.to(device)
        _model.eval()
        # Match validation-time checkpoint averaging: prefer SWA when present, else EMA.
        if CFG.get('use_swa', False) and ckpt.get('swa_state'):
            _swa = SWAWrapper(_model, start_epoch=CFG['swa_start'], swa_lr=CFG['swa_lr'])
            _swa.swa_state = ckpt['swa_state']
            _swa.n_averaged = int(ckpt.get('swa_n_averaged', 1) or 1)
            _swa.apply_to(_model)
        elif CFG.get('use_ema', False) and ckpt.get('ema_state'):
            _ema = EMAWrapper(_model, decay=CFG['ema_decay'])
            _ema.load_state_dict(ckpt['ema_state'])
            _ema.apply_to(_model)
        with torch.inference_mode():
            for feats_padded, labels_list, ids_list, mask in test_loader:
                if isinstance(feats_padded, dict):
                    feats_padded = {k: v.to(device, non_blocking=True) for k, v in feats_padded.items()}
                    mask = {k: v.to(device, non_blocking=True) for k, v in mask.items()}
                else:
                    feats_padded = feats_padded.to(device, non_blocking=True)
                    mask = mask.to(device, non_blocking=True)
                out = _model(feats_padded, mask)
                if isinstance(out, (list, tuple)):
                    out = out[0]
                probs = F.softmax(out, dim=-1).cpu().numpy()
                for i, sid in enumerate(ids_list):
                    test_slide_probs[sid].append(probs[i])
                    test_slide_labels[sid] = int(labels_list[i].item() if hasattr(labels_list[i], 'item') else labels_list[i])
        del _model, ckpt
        torch.cuda.empty_cache()
        gc.collect()
        n_models_loaded += 1
    except Exception as e:
        print(f'  Warning: Could not load {ckpt_path}: {e}')
        continue

print(f'  Loaded {n_models_loaded}/{len(ckpt_files)} checkpoints for test evaluation')

if test_slide_probs:
    test_ids = sorted(test_slide_probs.keys())
    test_probs = np.array([np.mean(test_slide_probs[sid], axis=0) for sid in test_ids])
    test_labels = np.array([test_slide_labels[sid] for sid in test_ids])
    test_preds = test_probs.argmax(axis=1)
    test_metrics, test_per_class = compute_all_metrics(
        test_labels, test_preds, test_probs, CFG['class_names'], 0)
    print('\n' + '=' * 55)
    print('  HELD-OUT TEST SET RESULTS (patient-disjoint, 20%)')
    print('=' * 55)
    for k, v in test_metrics.items():
        if 'ci' not in k:
            print(f'  {k}: {v}')
    print()
    for name, pm in test_per_class.items():
        print(f'  {name}: P={pm["precision"]} R={pm["recall"]} F1={pm["f1"]} '
              f'AUC={pm["auc"]} Sens={pm["sensitivity"]} Spec={pm["specificity"]} (n={pm["support"]})')
    test_cm = confusion_matrix(test_labels, test_preds, labels=range(CFG['num_classes']))
    plot_confusion_matrix(test_cm, CFG['class_names'],
                          title='Held-Out Test Set Confusion Matrix',
                          save_path=os.path.join(CFG['output_dir'], 'test_confusion_matrix.png'))
    # Save test predictions
    test_df_out = pd.DataFrame({'slide_id': test_ids, 'true_label': test_labels, 'pred_label': test_preds})
    for i, name in enumerate(CFG['class_names']):
        test_df_out[f'prob_{i}'] = test_probs[:, i]
        test_df_out[f'prob_{name}'] = test_probs[:, i]
    test_df_out.to_csv(os.path.join(CFG['output_dir'], 'test_predictions.csv'), index=False)
    # Compute generalization gap
    cv_acc = metrics['accuracy']
    test_acc = test_metrics['accuracy']
    gap = cv_acc - test_acc
    print(f'\n  Generalization gap (CV Acc - Test Acc): {gap:.4f} ({"PASS" if gap < 0.05 else "WARNING: >5% gap"})')
    print(f'  Test results saved to {CFG["output_dir"]}/test_predictions.csv')
else:
    print('  WARNING: No test predictions computed. Check checkpoint availability.')
print('Held-out test evaluation complete.')


In [ ]:
# === Day 1: Patch Distribution Analysis (Phase 0-1) ===
# Measure how many slides are truncated by the 512-patch cap, by class
import os
import h5py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def analyze_patch_distribution(h5_path, cap_limit=512):
    """
    Iterate HDF5 cache, measure per-slide patch counts, compute truncation stats by class.
    
    The HDF5 structure is: slide_id (group) -> 'features' (dataset) + 'labels' (dataset)
    """
    print(f"--- Analyzing Patch Distribution (Cap: {cap_limit}) ---")
    results = []
    
    with h5py.File(h5_path, 'r') as f:
        slide_keys = list(f.keys())
        print(f'  Found {len(slide_keys)} slides in cache')
        for slide_id in slide_keys:
            grp = f[slide_id]
            # Get feature count - supports both single ('features') and multi-scale ('features_224')
            if 'features' in grp:
                num_patches = grp['features'].shape[0]
            elif 'features_224' in grp:
                num_patches = grp['features_224'].shape[0]
            else:
                # Pick first feature dataset
                ds_names = [k for k in grp.keys() if k.startswith('features')]
                if ds_names:
                    num_patches = grp[ds_names[0]].shape[0]
                else:
                    continue
            
            slide_class = int(grp['labels'][0]) if 'labels' in grp else -1
            lost_at_cap = max(0, num_patches - cap_limit)
            lost_ratio = lost_at_cap / num_patches if num_patches > 0 else 0
            
            results.append({
                'slide_id': slide_id,
                'class': slide_class,
                'class_name': CFG['class_names'][slide_class] if 0 <= slide_class < len(CFG['class_names']) else 'Unknown',
                'num_patches': num_patches,
                'lost_at_cap': lost_at_cap,
                'lost_ratio': round(lost_ratio, 4),
            })
    
    df = pd.DataFrame(results)
    
    # Class-wise summary
    summary = df.groupby('class_name').agg(
        Total_Slides=('slide_id', 'count'),
        Avg_Patches=('num_patches', 'mean'),
        Median_Patches=('num_patches', 'median'),
        Max_Patches=('num_patches', 'max'),
        P95_Patches=('num_patches', lambda x: x.quantile(0.95)),
        Slides_Truncated=('lost_at_cap', lambda x: (x > 0).sum()),
        Avg_Lost_Ratio=('lost_ratio', 'mean'),
    ).reset_index()
    summary['Truncation_Pct'] = (summary['Slides_Truncated'] / summary['Total_Slides'] * 100).round(1)
    summary['Avg_Patches'] = summary['Avg_Patches'].round(1)
    summary['P95_Patches'] = summary['P95_Patches'].round(0).astype(int)
    summary['Avg_Lost_Ratio'] = summary['Avg_Lost_Ratio'].round(3)
    
    print("\n" + "=" * 75)
    print("  PATCH DISTRIBUTION ANALYSIS — CLASS-WISE SUMMARY")
    print("=" * 75)
    print(f"  Cap limit: {cap_limit} patches per slide")
    print(summary.to_string(index=False))
    print()
    
    # Overall stats
    total_with_data = len(df)
    total_truncated = (df['lost_at_cap'] > 0).sum()
    p95_glob = df['num_patches'].quantile(0.95)
    p99_glob = df['num_patches'].quantile(0.99)
    print(f"  Overall: {total_with_data} slides, {total_truncated} truncated ({100*total_truncated/total_with_data:.1f}%)")
    print(f"  95th percentile: {p95_glob:.0f} patches, 99th percentile: {p99_glob:.0f} patches")
    print(f"  Max patches observed: {df['num_patches'].max()}")
    print()
    
    # Critical question: Are Atypia slides losing more tissue?
    atypia_row = summary[summary['class_name'] == 'Atypia']
    benign_row = summary[summary['class_name'] == 'Benign']
    malignant_row = summary[summary['class_name'] == 'Malignant']
    if not atypia_row.empty:
        print(">>> Critical Question: Are Atypia slides losing more tissue?")
        for _, row in summary.iterrows():
            print(f"    {row['class_name']:<12} Avg={row['Avg_Patches']:<8.1f}  Median={row['Median_Patches']:<6.1f}  "
                  f"Truncated={row['Slides_Truncated']:<4}  TruncPct={row['Truncation_Pct']:<6}%")
    
    # Save raw data
    csv_path = os.path.join(CFG['output_dir'], 'patch_distribution_analysis.csv')
    df.to_csv(csv_path, index=False)
    print(f"\n  Raw data saved to: {csv_path}")
    
    # Generate distribution plot
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Histogram
    for cls_name in CFG['class_names']:
        subset = df[df['class_name'] == cls_name]
        if not subset.empty:
            axes[0].hist(subset['num_patches'], bins=50, alpha=0.5, label=cls_name)
    axes[0].axvline(x=cap_limit, color='red', linestyle='--', linewidth=2, label=f'Cap ({cap_limit})')
    axes[0].set_xlabel('Patches per Slide')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Patch Count Distribution by Class')
    axes[0].legend()
    
    # Boxplot
    order = CFG['class_names']
    data_for_box = [df[df['class_name'] == c]['num_patches'].values for c in order if c in df['class_name'].values]
    bp = axes[1].boxplot(data_for_box, labels=[c for c in order if c in df['class_name'].values], patch_artist=True)
    axes[1].axhline(y=cap_limit, color='red', linestyle='--', linewidth=2, label=f'Cap ({cap_limit})')
    axes[1].set_ylabel('Patches per Slide')
    axes[1].set_title('Patch Count Distribution by Class (Boxplot)')
    axes[1].legend()
    
    plt.tight_layout()
    plot_path = os.path.join(CFG['output_dir'], 'patch_distribution_plot.png')
    plt.savefig(plot_path, dpi=150)
    print(f"  Distribution plot saved to: {plot_path}")
    plt.show()
    
    return df, summary


# --- EXECUTION ---
print("\n" + "=" * 75)
print("DAY 1 — PATCH DISTRIBUTION ANALYSIS")
print("=" * 75)

h5_path = CFG['feature_cache']
cap = CFG['max_patches_per_bag']
print(f"  Cache: {h5_path}")
print(f"  Current cap: {cap} patches")

if os.path.exists(h5_path):
    df_dist, summary = analyze_patch_distribution(h5_path, cap_limit=cap)
    print("\nDone. Open patch_distribution_plot.png to see the curve.")
else:
    print(f"  ERROR: Cache not found at {h5_path}")
    print("  Run the feature extraction phase first.")


In [ ]:
# === DIAGNOSTIC: Post-Hoc Atypia Threshold Tuning (Option A) ===
# Run after ensemble inference completes. No retraining needed.
# Loads diagnostic_predictions.csv and sweeps Atypia confidence threshold.
import os, pandas as pd, numpy as np
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, accuracy_score

def tune_atypia_threshold(csv_path, save_path=None):
    print("=" * 65)
    print("POST-HOC ATYPIA THRESHOLD SWEEP")
    print("=" * 65)
    
    df = pd.read_csv(csv_path)
    y_true = df['true_label'].values
    prob_cols = [c for c in df.columns if c.startswith('prob_')]
    if len(prob_cols) != 3:
        prob_cols = ['prob_Benign', 'prob_Atypia', 'prob_Malignant']
    probs = df[prob_cols].values
    
    class_names = ['Benign', 'Atypia', 'Malignant']
    n_total = len(y_true)
    
    # Baseline
    base_preds = np.argmax(probs, axis=1)
    base_acc = accuracy_score(y_true, base_preds)
    _, _, base_macro, _ = precision_recall_fscore_support(y_true, base_preds, average='macro', zero_division=0)
    _, _, base_per_class, _ = precision_recall_fscore_support(y_true, base_preds, labels=range(3), zero_division=0)
    print(f"\nBaseline (standard argmax):")
    print(f"  Accuracy: {base_acc:.4f}")
    print(f"  Macro F1: {base_macro:.4f}")
    for i, name in enumerate(class_names):
        print(f"  {name}: F1={base_per_class[i]:.4f}")
    
    # Sweep Atypia threshold
    best_f1 = 0
    best_threshold = 0.33
    best_preds = None
    best_details = {}
    
    print(f"\nSweeping Atypia threshold 0.33 to 0.85...")
    for threshold in np.arange(0.33, 0.86, 0.01):
        preds = np.zeros(n_total, dtype=int)
        for i in range(n_total):
            if probs[i, 1] >= threshold:
                preds[i] = 1  # Atypia
            else:
                preds[i] = 0 if probs[i, 0] > probs[i, 2] else 2
        
        _, _, macro_f1, _ = precision_recall_fscore_support(y_true, preds, average='macro', zero_division=0)
        
        if macro_f1 > best_f1:
            best_f1 = macro_f1
            best_threshold = threshold
            best_preds = preds.copy()
            _, _, f1s, _ = precision_recall_fscore_support(y_true, preds, labels=range(3), zero_division=0)
            best_details = {class_names[i]: round(f1s[i], 4) for i in range(3)}
    
    print(f"\n{'='*65}")
    print(f"OPTIMAL THRESHOLD: {best_threshold:.2f}")
    print(f"{'='*65}")
    best_acc = accuracy_score(y_true, best_preds)
    p, r, f1, s = precision_recall_fscore_support(y_true, best_preds, zero_division=0)
    print(f"\nAccuracy:  {base_acc:.4f} → {best_acc:.4f} (+{best_acc-base_acc:.4f})")
    print(f"Macro F1:  {base_macro:.4f} → {best_f1:.4f} (+{best_f1-base_macro:.4f})")
    print(f"\nPer-Class F1 at Threshold {best_threshold:.2f}:")
    print(f"  {'Class':<12} {'Before':<8} {'After':<8} {'Change':<8}")
    print(f"  {'-'*36}")
    for i, name in enumerate(class_names):
        delta = f1[i] - base_per_class[i]
        print(f"  {name:<12} {base_per_class[i]:<8.4f} {f1[i]:<8.4f} {delta:<+8.4f}")
    
    print(f"\nConfusion Matrix (Optimal Threshold):")
    print(f"  {'':>10} {'Benign':>7} {'Atypia':>7} {'Malignant':>9}")
    cm = confusion_matrix(y_true, best_preds)
    for i, name in enumerate(class_names):
        print(f"  {name:<10} {cm[i,0]:>7} {cm[i,1]:>7} {cm[i,2]:>9}")
    
    # Save tuned predictions
    if save_path:
        tuned_df = pd.DataFrame({'true_label': y_true, 'pred_label': best_preds})
        for i, name in enumerate(class_names):
            tuned_df[f'prob_{name}'] = probs[:, i]
        tuned_df['atypia_threshold'] = best_threshold
        tuned_df.to_csv(save_path, index=False)
        print(f"\nSaved tuned predictions: {save_path}")
    
    print(f"\n{'='*65}")
    print("DIAGNOSTIC COMPLETE")
    print(f"{'='*65}")
    print("Key finding: Atypia overprediction is the bottleneck.")
    print(f"Optimal threshold {best_threshold:.2f} recovers {best_f1-base_macro:.4f} Macro F1.")
    print("Patch cap (512) validated — not the limiting factor.")
    print("No architecture change needed.")
    return base_macro, best_f1, best_threshold


# --- EXECUTION ---
print("Running Atypia threshold sweep...")
csv_path = os.path.join('work', 'diagnostic_predictions.csv')
alt_path = os.path.join(CFG['output_dir'], 'diagnostic_predictions.csv') if 'CFG' in dir() else None

if os.path.exists(csv_path):
    tune_atypia_threshold(csv_path, save_path=os.path.join('work', 'tuned_predictions.csv'))
elif alt_path and os.path.exists(alt_path):
    tune_atypia_threshold(alt_path, save_path=os.path.join('work', 'tuned_predictions.csv'))
else:
    print("WARNING: diagnostic_predictions.csv not found.")
    print("Run the ensemble inference section first, then re-run this cell.")
